# Qxern v6 — public independent audit

This notebook reproduces the public, one-shot evidence summarized in the [Hugging Face Forum audit post](https://discuss.huggingface.co/t/qxern-v6-two-llms-that-talk-in-latent-space-32-tokens-zero-code-text-a-symbolic-sidecar-built-in-one-week/178411/6) and the corresponding audit section in the [Qxern repository](https://github.com/bezvka/Qxern).

It is an independent follow-up using the released Qxern v6 adapter. It is not an official upstream benchmark unless the maintainer adopts it.

## Included

1. **Causal channel replacement** — separates sidecar-driven exact facts from sample-specific latent semantics.
2. **Identifier accessibility and candidate-free receiver response** — distinguishes encoded information, external decodability, receiver likelihood response, and exact generation.
3. **Pairwise behavior-state sensitivity** — tests localized code edits without showing the candidate behavior descriptions to the receiver.

## Intentionally excluded from the core notebook

- The earlier closed-set identifier test, because the candidate-free test supersedes it.
- Exploratory history/state-stress notebooks whose fresh-state gates did not pass.
- The 0.8B→2B linear-bridge experiment, which is provided as a separate optional appendix because it is a cross-receiver compatibility test rather than a valid receiver-scale comparison.

## Reproducibility boundaries

- One seed, one released adapter/receiver pair, and small evaluation sets.
- Python only.
- Semantic scores use embedding similarity, not execution-based correctness.
- All model, adapter, and dataset revisions are pinned inside the experiment specifications.
- Run in a **fresh Colab GPU runtime** from top to bottom. The notebook uses isolated worker processes to release GPU and RAM between model phases.

At the end, the notebook downloads one ZIP containing the three result archives and a checksum manifest.

## Executed snapshot

- Executed top-to-bottom in Google Colab on a Tesla T4 on 2026-08-04.
- All 17 code cells completed without an exception.
- Public result bundle SHA-256: `2fefa1e006f4644edf598aea26c72a0dfdf1420450f060c461686c220ddc88f9`.
- Rich image outputs, download JavaScript, progress widgets, and repetitive worker logs were removed from this snapshot. The summary tables and completion records below are retained; rerunning the notebook regenerates the full machine-readable result bundle.


In [1]:
# Install one pinned runtime for all public audit sections. Run in a fresh Colab GPU runtime.
import subprocess
import sys

already_imported = [name for name in ('transformers', 'datasets') if name in sys.modules]
if already_imported:
    raise RuntimeError(
        f'Packages already imported: {already_imported}. Reset the runtime and run from the top.'
    )

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
    '--upgrade-strategy', 'only-if-needed',
    'transformers==5.14.0',
    'datasets>=4,<5',
    'accelerate>=1.10,<2',
    'huggingface_hub>=0.34,<2',
    'safetensors>=0.5,<1',
    'scikit-learn>=1.5,<2',
    'pandas>=2.2,<3',
    'scipy>=1.13,<2',
    'matplotlib>=3.8,<4',
    'psutil',
])

PUBLIC_RESULT_ZIPS = []
print('Pinned runtime installed. No restart is required.')


Pinned runtime installed. No restart was required.


---
# Part I — Causal channel replacement and external identifier accessibility

This section measures exact-fact sufficiency, sample-specific semantic use, packet distances, and grouped linear identifier accessibility.

In [2]:
# 2. Experiment configuration and isolated worker

import base64, gzip
import json
import os
import subprocess
import sys
import time
from pathlib import Path

WORK_DIR = Path('/content/qxern_causal_probe')
ARTIFACT_DIR = WORK_DIR / 'artifacts'
LOG_DIR = WORK_DIR / 'logs'
RESULT_DIR = WORK_DIR / 'results'
for directory in (WORK_DIR, ARTIFACT_DIR, LOG_DIR, RESULT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SPEC_PATH = WORK_DIR / 'spec.json'
WORKER_PATH = WORK_DIR / 'worker.py'
SPEC_PATH.write_bytes(gzip.decompress(base64.b64decode('H4sIAMRScWoC/2VTy27cIBT9FeR1ZoKxjXE3UZMuuq266KKq0DVcxigecGw841GUfy94YnXSbCwd7nnAAb9mE6LOvpCS3ZHMSTxBH1GekIYAEwZp0zw7eH/o8V55jXKRh35GqYJcYfAy4BKyG43yzthD0g2X0Hn3YQZO2whQDt6vaZzG8TFa9RLe436c0d2nD9tXu6c4Gnf5vnrMbogjnuxko3eka1Mo5LWiyMpSUMNoyUXL0TDRlJwxzRts66L9p2//DypiEN2Lx1vKbQQzivKCl3VetQ2vTS44BYRGmULUtRB1VQnR5nXSt7PTPUb54JMUFnu09y8Ljm534juNQ+8vO+tC+YH8LwtrSsEwbeqW56LKochLZtIRkNaqRYE0Lwxfs0DDEHCUxvbo4Ihr4HVtumbKjVKw4J/3Q7iVTR2wiidRK6qqYjlFQTWjXBVQI49FFoWmDWiOrKKxzJpHpHRZi0aVgpkynpw2tRJrcbBsb+IZ3RRto3lcn6xGBaPs8YTpyrOfLNEnPIILVsmXGafwfvgncOTiZxKw78kRybmDQEJnJzJYVEi8ISmCgFL+OPR26nB6SG64QHySm1UKf80GGOEYtzS7kLy/+zOJkReyrmPqiGiP09XfzE4lKQnwjHckvtNrOIwYCXhZUzaS3Mr+lSg2WSBJa2mDH+xW2YhhHt0k4w82r7Jvn2OvHAJkJT2Qr24640gukehH4vw+e3vv2OH5vWK5dZh+YvF5vJaS7qGMs2H0LUrnncJ1+6mj39nLcpLQG1ifUwLtCCcvNqQ6GPtmQ/HHCMA3hKrz1QaMX64vOoGD702xgc7Hy2TZn7e/CKe07nAEAAA=')))
WORKER_PATH.write_bytes(gzip.decompress(base64.b64decode('H4sIAMRScWoC/+09a4/bRpLf9St4PixIOhxmxk6CQIiMm7Wdy2KdxIknCyxkgaDE1ogZimT4mMdO5r9fPfpJUvNIsnt7wBmwRXZXV1d3V1fXq+nZtqn2XpJs+65vRJJ4+b6ums5Ly7Lq0i6vynY2U2XNeZ02rdDvbaced2m7K/K1ev25rUr1XLXqqUnLrNrrN42ny/diRnTUaYdoFBHv4VX3Xvb7+gb69MpaFdVt3+WFRlM1m53zEpclNdC0dEBCu62avWha7jFLu7QVXau6LKo0S2QhQ+z68/O8PN+mG5Hsek3bN9vTOo+83RYLk6y6KrEpN7G7UfCnfVe9rsptfh7R87dVJoqvq+Z12rdp8e5bLj2rLkSZ/0M0s9ksE1uvLdO63VVdUKRrUcy9tmtC7+iV911VivnMgz+Xe28hJyK+zJuuT4tkL/ZVcxOEBFA31caAvIc30bZB1cbnoqvzLAglWHqDAwDIW3rHPz716s89+o1MeXvTdmKfpJdpDlWFSM7XAHW5j3WJ96l3cvzis+fPX1rNau48aeAvtcCCmKlN8nJbBWEMdZNtN32WJmlRVJu0Exk350XGGoVEAwShweLlWxs0bw3hACaKVtB8DjtrRCuay8N9qfrf3NWdXJ+87ALcMXEGLN4GciXCyNsWfbtbnDW9CBU/7NIXn3+RbHPAh5uFOML7lXYKMQa8Ml9k+bloO1hOuTVjbiqZ4irvdl5Vi5KwRJ7frP0Q98oO9mghWQv/ABd7m11fXnh56eWdaIAT9+ssnUvIuBFpFuDovec0CUD22vdDg8EQE/c17CwRED6moxEgdkpVvxPX/BTgeDdFCrzwDlqU3Q9f024KyjKGjdMXQnaAc5IA6+RdkgS6y1YUW7OaWb4HiDkMoHMLq74blIKMSTrcgu0QXNQ42SMUuzzLxBD3dltOV0AxzBbjhpX50sLVVDWRs4Wlx7rj+IRrrZls+xqmIIz1gENdBWxnyPH+pHtyVkEWAvZSXHfBjhZ3hwsbnHwRATneZ5H3IvJOwhG+nbcAmkJnhuNfetHkgvCV8fu0SfcCGYT5H4V9GZgJjSyEIfDKcXz8YoAPpMHPMDTG9y4vRdoEvHhOY7fRGjb8haSBWeNdTgxkc3CCgwSKzkVAKzlgTwtRnNawKzLDaG/yTRfcOuAsIpqqbUEuIGBfdDlO7WmHvApnZjCCdxkmUmsRqZVfyF/YPWm32cEOb9qOtv4IVRiNqSlhe5wwNe/SG9F8B++BNWcTTXDQ/1b0v3ga/bDLuMEHAYwIZKfFNNkuM6kBmE06gVw2+++3734KDle/4SEHcuiHAWX3pkuHn6NHz9DLR8/QXTi1t4DI0eayCcJBuM2wV6uZ0zFBaykM2+wqbbKAZK+3Szag36hT8wz2f9VEXtp1ZbJP2wu3gg4uu8DsTj5qoXdbQASMPSZZGVgjRYGkgaV0ivuyhUfxDxEch7G4rkEuKQRtTqWRd3SCfw2iOs0y0PmIVsCn6SYp6EgWkhooXSwZ4gqXFDdVJrIIpNCC4ZdSeqzGDEtDiOS4ze+FuElsohb2i4MknI3QmW5ZTKwCLv5E0xY+TDGJC9ky8uyfB3t88WCPE41eWo1kMe54WWgtulQhHHYNHJ7XTViHIhU/zdIaD6u2Fps5MP8GJBcdVEqjIjUNOPsy32g+5jd5eEgMCSpQqGa5hkBg0VdXSZ4tsKelv+5Ra0qw0F9ZKgModCWcnxJK4VbFNmgDRLQgo4cIudSGpPEs6F+pSzDhGzISQB/ErWIplPaIGBTUAAv6PwDcIY8b+yvD8E2ag4r7Y1+iTfe2aarGZfHts1NuC0qlgK3S77193u7xyJh7t6avO+zsdrq3u2czl9m75sZSksBqFTAwXjFaCntcsKXSOiEjAefQ39S9H3lXIj/fdW1SlcWN1LcRl7jegLbgnd3UPJbf1YueUVinEtqXGxEQmojYD7TvMoNTGUsSLPBJqlA/WDNstbRBVxLHmMAJWILZFCItBZl7bIagMAMpE3mXadEL3XcMOv++DSzM1BLaAbAjCetGbPNr0ib9PalOMcysWj16/oVt4nhoHsCkENYYumy6Fo2TgNENAO3+6XdZoAkjQVcujSJb0u8KgGlQM3tiJMhMmiG/9HlD0xH48uBAguWBEzN/6BKQKVaRVByPY5BP8UtVwcsNzA3rdg54lzBfapJJG1VdwuCxqKw6PekrxSqy+XB//VXc8N7a+s52qis0LFSfgBVMjVv5eufLw3paJTesooa/QouxFjOl+aG0QEtVgg2mRkIvT1ahbrGeaGFNnWpyLJvQPIKkRGV+ubKZUk+MpRigyADARsT0GDRyHT7Gwcfsk/AjMhw0dUwkljMOR+lOleKPBBNcfN5UfR2chPKsIcMBetyn14FuFcLpdDJzjT531A53HE+N3T5NoK1r8A6s2AWtxMiMpeL1lB27sJbbtWYX9O+UObuw9MIJo3Zh6bF8rMRdFfDZCCzVgbRcsFwkBe3lC2Xqt2BnoDYlJQKdw0YwKWEIxy+8WVJYwQvYxNKuI3mjDE7kDgWjCx2RpUtjueva5LyBzoOv06IVjidCIuKtkfLPOrJOwUgORCoT8FIVlyLpwKIGpRL9ewH/yP7PRcfrCg+g/DSyFuQGFNjNgF9R59DHxCYt2G/EGELXJ3MByvZ5S9L21s8EqrMNWAc4a3cRyPSB4HSOSPWHJf1Ckhg8f85IwxEgUCNPhZbklPE+Dv/IWTTiVv05fJSaRWpbFlCi7UgUj2bs0GzJXmVDoFY9GXLZ98Zt5dIxCydkBMglQ0ME9i4TZ/VGgubgQs8Gk0lEW3BAudXXcJ31zLqzij5oPKYIA2ww3iNmbVR/CIeu3MDpwxF7B3qwsQQGjT3JoQesBsx0qIPDyPmgOoWZyNd9J9Rx9ZriCmoy5RJ4iNEjz/ktSg81r3GSoPKbJHe+s84oXalbpdHXTb5Pm5tk25cbcl90jQClPW27+PTDmdwKXQUambgUBR4vRlLCxqENRQ+wnfT6AQo83KvsBlZsuXJm1FLGsF3kBdjX17L7NwIsYOq8vSk3Vqk8SVb2YDRZcBYgalRodFmIBueJ8Rsrzk3bhFk8aTcVTNi2NKNFHl5XVTHXklINDSGu0uICwC3xgNuFAFoP0DjcAevQ5aW1k3/70Ln0Nbp09ds78iSHA1El50W7juQ7iWrlBc9B4qVgHoFYbgOUfbQtOTwyH9sEuJZ48ECfFMAyTRw9P/hwU3bpNfFq5P0NOYyeLQIlMbQWNL142k+yn94gACJ3B26miaW7Bz3J+AWgiPFpps8yLDSqwRJ1dISg1cYXm5GxJSquVYvWDb4xPxtt+ZMDGBCWOrZBg6X/3EdDHisvgZjmHGGId60y5ll729zby8WVIm7Yl+mMTie3Lypyu+J/cZrlZMK6lzdmskb8i1zxI4GyCYal8ei4cw7fqQ1ldU6yb6DFPrgHp+lCTBZZwKiHz2CwnWDN0VMnQWNkyAncWCf3I+gYMiylKoaGmdWWO+CW34FgZsK4NM4zZcLw8McHPJcrPVs3c0S7FXskv8cceR+f7Cgi7QCMHtKDVVMmuo7sQnqxPKO+YQt/bvGIBcFEQi0/LOcvVypYpxS+MkNnBIuggEQQO5C004hEN/uRFp7/4YXvRuZQJmCriUOTJ8H31U7vWuew2j5DCm6p9ZInaHUH+h9s7/hnsP0CWSNnYRXehc8sDf6Z0Y7nBouas9WdhNXWJ5+WZNDDIKCTDy99V6vu9HI6q+3zQFovZfVg7t3AnOqBL+11WDH7jZuUlT8bOzUNDrlSK5fRHKIkzNxDGTKaJYUgdDjQh2WUcITLRCLVafYjueQaOve+g11zZqL9U1FJdohXRUbKDLFFBArqlXm13Tbot1SwsPbqceCVl60pksePxhGPfsAusY5eSQFu8PGxHzOehdvzIG5o4Nz+XarORSnAdEuIABI0IyctFjKlTOVQS0BPzZD84chQ8tw/JBBFjxgQQh0YjkOt2vWoieoT3tU5huvpbPeHlI8nqRBDBdsoKYH/9roWGzR7xDUweHHjocoBquQR72OFWurSFpNJEWtTO+R1BW1GGsa8zoZQHOE2v06kw0m7P1sLRlnZmAxQ8nRwJU8z+e9xfgIOH40mU23Sj6Xco0VeihhdBnUQ0kGLBSizOMDDFXFbF3mHNW1AsW27VaiXuK6Sdb+5AFMHn+XKcomO2Z8cH7tWomWTqEyLffY5YYhFSWPx+2579KUfhnaCQ+SdfBF6f1L4lWYrCljDpE33dQHEjqITmBo0EahQIYk6BxIpMymwnP/Sf8VJTTpSgDxZ57EqpvQbdrjrogwdyzYi9FpJL2ojUsw8svOlrASMMZ4DddLQdOphrRZ+Bwxe+lYx9Qh8RZ6hiXDIcHx2yGMDikqOuSeWVtYK9nqKznIqKaWUe7Oc3qi+oYu0CRBk6WOBb2m3GSVZmXp4R/4qYWihy4YBMC+wbwj2nqpwYkUWGja7KVZElrjflxdldVW65v2QcUPvFfLpw6Yc6msBKko4GGgFjUijg5+vSIGCQYTw+BJ4/gFs6KQ1qUa4AdytHN67F2yilLsXVueBPhEkTrMscNy8rFotJmzEsRIx7RMZdUS8D2PpSQKwgS73IiEGjTgDcw74Ok8L5Xcl/2Wcl3XfkaPY7txG+EpuiH16nfDJQO1BLzryvnyAMMPWSuNxE1WYS+fEvW5+gMWfc+TdaKi/AcvNibMGNTRvqJbj76BOTjk1HCvJkRyoWhg6lWB3RGOS1RzM7ZkycHfOZBID64kg5h+IGFWZ1FVVDBXGNWz0i9kBXF9JVGWCzuenBTq/B4PWux1gvPPaPu8oQdKSSVeiQSdUD3uPbEXqbXHr9j0KezYUV+KkWrAU8UcKcNgZmRJOAAVSu99uC2ETYp9dpng5d/vEuOJArMJB9V/s3YcTA+gGwzDZ484O6QDjbQ42BZ1qAdYUztmlHewIB8dqkbfdElhxhTHTayf7zlMxBU4pQi+hOoY/kwKJUgkHpjYFEU3C13HkKRnXhjYuS6atZTSJgJaEYM5oPrEaGNcEbeuW4r9KELhMhW2iCbcW6H2Y4NIu/Lrzo4HlQhkdC5IEQPF5txtAdA1oZRxRds9ADoipVgszjwbEitGY3VP1XU25PbRM7hCeP+dBur1wi0R5gjFi004Q07cCtt1mJxbktpuciUxFeaIJM09H0rjD2OlweXSyUsk/ljVKHGfiS4HKbOJxLP1UZbdR0ox9eDMXKRkqMcWbuocze5cW28A1EWWCL3TPDSlKtDg23mixLTDK91ByiZxMzuOlDF5lPuIy6ewCfg0oKXl+LM9+nRju6+6OiGN9J2g2imOZXIqIdTcdlhulxUia1TaUmhTpaxQMNhntMfruEyCElDaRDRVBYq8klQrlIIFFVVoZLE7mCrJ9i/JnX4EIxx1q0godmta/gab1fTStfwdN+VR8KUlDzGvhaDo6g8cQaw2xfsph8wYqStLqTTrNqXc7ScLdp7dEwV3k/XkSZK1A1qMzB4ZlAiTmkkPk+T9cifJl8rl1o0EFug7Zrn+hixHSeB03x2Z9aS4UgEDvdlB2Zl+uWPd5kfn33GBAidGjJuFXF5Y4HSYVzZ2dYcBoolBFcsPuVL6W5WvbA9nvk4IC+EaHQdNQxadVUqKO/k+QxBkD0oepGnLhFLi1i32SL4FVYjewFyy5hH+QsefOZZU4URVJYrU8r3v3BgSGrlkqkC8AcyityxPVHsQaHgB1us7B4rnx+ZgPpjEYOMAT2lcjLAEZXzVgEZHbZOKuBAp54OJu8QJOeFJB6CSVtsbTr1mA2KjTRvzLBDh25ohvrVdImabvBf0vi1pNVwzqCrO4rQXFomq5lMfIboxoyvcw4ekY+zgOLJ21v4eYcU9OOgJUzju5yFS3cpPJN5uJ0fTR1WBYs1XPaZuup0C2VYw75KfHHsbJo49guYqSM4b3yJ7EIIeSSicY5XBWqVRNJxjH1BfVVQL6FN5MS4DO82G9FAX7tF7c+jDtx3dW3TCj6OQLU0mp2TkuARyAnUy3bLM69VVukpU2hOEsTO/i0ZEMYgNd7NeCtO8WND5O0ZIiGlRNc2bfeyL7tBJw6l7lWbcDo1pwoI4z1VR+F/V3UAQQZUfpETKCyFTGXtXk53mZFjSxZOzYPqgpblzNXJV4YJWZLDqeiWkDze02OuSnUMwppYRMhmnytOwMwSunmK4TTpRTxt1EOZ4xAyOPQIGJIz10AWevaPB+mdqSbsYU32GEJSnlqTVoRKOrm2oNtRoGw0PDjBlraMpkGIYH7PWxuwzDSVw8HwoZvU0D8gQpQDUH07BEvr7OZEhwZvZ3sogzGU/ikGllTUtldXL8VkEvh0FBYnqy6vYiLeU4Tb2yJ2OsDdiek7af1XQwbXjn1S2ZgOXF1foTOwACt9ZNmyyq8nyqV179Q5i49nGYmLXnLq9MwNG6WnC8zv9MLfpOqhlIfZteCqPhWcrWw2oBNs2kOmw3HLFIQmJeKaiKGajw8MK7jQYs4DQeqQOzD3//cPb22+T9j99/+/5Mp/E8+3vVe3ACeKm3E0W97QsOOaRtm2P2RRd7bIo9Oy3bKzhHwJBoKbjmpWu8n9XtBHpdGr+VDTebHuVacUMufbDsNnBiFTfxs1k4e3d69va7s+T9u9PXb7/5/t2btz9iosJXX33FFR9evXrlzz785c3b16c/TkDJGoJykiI2u7QLLDkBTIVnfTudk3UzmfdldEkQXcUN4QS1H8QBSulRUonqYnxjTiGackGxUpYlFLImpQE0JjBbuglfFiWIlXTLHezP8gItiwmURgjfn+r6O4c5HNY9w5ggT5L2ln4AfJR6MsjjkEHWETnbZ1/9CnuWLJZfX91K6kA/rgpMSflYmiIMWoBmBqXUBBjl11cfy2djlHBEy0Z4MKsB33O37RNgRosKvVU+lnbiCDMo3fZIUto7gckVdGLL7Duuasp4MVv/Y/mD3GtzTCH5WL7G0DQ+mgHpN0mL7Wn4WOK+BKKo8U+4R/FajKKWmhJjMZZP1fPw9jfYI3nHthMOIN6CyRsgvU40REOBznp0MkgQg2ay9XKuAN1kVESsgpI8cxyBybPW3thW5oNzJ27I3GYasUU06QSfCp5xsifr7zqEZnuw5U0+9rTQ3qFkHZtGJSGn6JQzcUBsmYSrW+boufriBa6QYui554hyy2AZBOEUChLPo2iXQjaxyfxvMC6UtyTcSaoH6FtpYGuILJzDLhsL8jub+w0mzcLerZqXOxfOOu3kUFbqmGW3S4S8xTcWFAtyPHtMRDhihiEDGaRyXaIDcKpLYwLLFOibdZNn/7/0Dy/930STb3ORyZA3pdmnjJMScBrgiYmz/p/JSA1/meQRPLSHNR4zH7aXDScod7gvGETxH+TD++ENPY+DH/OvSku5P4wqT3Shjizj+IlcPxy/ugxvGf/uRXvluGTvA8fhHY+m3CGshVjI0KTDxK/Rt1FkhHYmN5f2o6iY4gEfy0z7ddDyVBXK80IVPIhcbGzLH3MnJHWY4ufzAeBbhySsZ46fpaCJ57uEjzolXBeb6VsZ0JrIQHYRuscuzfbh3GwXGUO7XyYYXaGjf8NH0CNHKkFFMZwkFpVTk4TcPJytRwnW/7uzJYfs0qO2wj1EGAyT+1xi0PPyW9atvS+7U62odJ9wTJv3VKsDDRiS5u44JC2/aOEGvjV0hemQg5QGiVE6P+cvVmN/ghrjwnHqGKudwjuY17trKpwdudtJM6ZrfegyjWvRbDmvBz9pJG9KsszLLOHBJQeoXDhvjk/YGu/CfY1mdsaEEWoL99VyP1cyWjC0+jDWtxbpvl2c2I70WnQ5m2JgNRbdzeIktup12AQ/CTERN0GX5mOnVBRpDepAwteBJqbVO1LTHtLXuY752yV8pqg78Noq0sTwddJAr8fyGNigvcjryRw3N7GXcEUWaRG53Ke4C6+KszKHLvpeOWHvibdJN3lmRdz+qCCcIsGJwqn+3A89OFQ84nsS1iVj6dakhG9GstROz6HP3kAob6Ob9GK5MR1o271pWvwRUcV/WrLE06KK3B/fyVg8PR1Bf1vBQvOHZCj4dnhurYQY9/CkCb4/Krf+N43KvXzxR0Tl1k+KymE7w/nsTrYDNofDdH++P0zHQuWwlOBA3XoQqAOTLsvZK2tf2/KVTlU1jdjYeX86GRYFhV1egd3XqEwS2U6l0/rDWMbDYKpKQsp80GwKtMbPwEq9+hBBZTVoybIrLYrE+KVhBgK/FfsUJn/j6+xfWaABYb1W3ifsSWcQur5iEIEQk9+gkbP8n97ZVeVdpc2e4h3ielP0Gdq3aNnKfEHt0NNLwvfYhivhjte+iEVSGfMUWDzDGThINUShy4LZrsMPN+gujfate3R1zEkzz/H78qZwom8HJzKSBI391zwInXuNoMosoO8+WqrchD/b0Uz/lWTzqa+svMOUsgZSXVmWYikz9VUs2Hh4qYAix3KNH4gdM+fTbbFAN8bPvuDtndK6IcJbbMQcus1qgHIEqXpajZDKddPsaK3kAOkQ0sJqN7J6kMya6JzmgQ9ryMDoibKHGg2pNLwVTl8QYGkHiHh9H938XqEI6OxpfQLa+2QoYLX0qScgfUjiTkyjs4CHMdsCawKLu2funUJLjI9m7yCau5mTVaG2cXKRl5lxENBVd/s4mI+S1Y2t5XzMYKAXOYCJPk5WU9+ocWgh4avhxxE9vHQ82QcdP4MO3MDT6Fwx5/7UZ214QvW6Gt/MaPMtdcF4fFOWFX/4TKb9ky78gFy2ZfNk5cCvOPF1Svb/TNYeEuOWt/FAr3JKpmsPGebOp2mlH3L0JdXxVwjgmJi+LTVIQ0FdfG4dF9OwctNaLZS8PdBALzHdy5LPB2AdhgZ4d7Pd38YCPwDJTIJJG8xa01AkATY3CX1VwWK+aWiHI+kD79b7gTbqVGirvtmIyXjIUKz6ltcON/pALB9sz58aoBUaonCPT8yZUM9TXwGZRlwCiD8JE943u92Th24cqK74f/TADQJb7D/QHE/CQWvrcPwj58a64TfO8qEE80KYNF57m6Jihl8Kq7oU/+OCcjIf90BeuR2zl5kPVr8gODC3vMXjN203eS4dO3QcQCXdOAPhgp8ERARmVMOEdNvBNzG8g7lLGE+8UknJ1NXdZPb6PgXKpQbLIT68FSX/8474tDnv0Sp/TzXqP4mgF7xNm6SyPvA7vCYFOs+uAtnaLpbmqhF/CZPsbnxUZrG6UzWJ7egIj1s/0l++tD7xd6DBVdVcHGV5M91IfhxJtqUfbN0qb5e84Xzwv2O4303w+qc3p9MOHhwG9EtrhvZ/G6DXkb6zFGNdSP8nArPU6C6C/O8XYGAJDAzQmLaqUI2uy9VdZg3+KUhtVe67YPH+AiDwmyUwefJOHtjGYFYn1YU1a8q7VrWxKC/zBgZBt8y/+To5+/6vb7/zzewRVZ387LW1+E4kybrqprPVDe2fWu1inDHfilKN8BM/Odj1LYyDuMn3E9edPxVG0Y7l2VC1HNysdP1m7vwP+rkHzORhtTTYwo+cTKQcPwvDn7SjIScJbtUkkWPmfTv7HyvqVlyTZwAA')))

# Edit the decoded JSON here before running workers when a larger first pass is desired.
SPEC = json.loads(SPEC_PATH.read_text(encoding='utf-8'))
print(json.dumps({
    'n_eval': SPEC['n_eval'],
    'nonce_classes': len(SPEC['probe_nonce_names']),
    'model_a_revision': SPEC['model_a_revision'],
    'model_b_revision': SPEC['model_b_revision'],
    'bundle_revision': SPEC['bundle_revision'],
}, indent=2))

compile(WORKER_PATH.read_text(encoding='utf-8'), str(WORKER_PATH), 'exec')
print('Worker syntax check: PASS')

{
  "n_eval": 12,
  "nonce_classes": 8,
  "model_a_revision": "df3ce67c0e24480f20468b6ef2894622d69eb73b",
  "model_b_revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "bundle_revision": "e700af2df7b61851a3142f9eb7e07cbe8e013f67"
}
Worker syntax check: PASS


In [3]:
# 3. Worker launcher — stdout/stderr are streamed to disk, not retained in RAM.

import psutil


def parent_ram(label):
    vm = psutil.virtual_memory()
    print({
        'label': label,
        'available_gb': round(vm.available / 1024**3, 3),
        'used_percent': vm.percent,
    })


def run_worker(task):
    log_path = LOG_DIR / f'{task}.log'
    command = [
        sys.executable,
        str(WORKER_PATH),
        task,
        '--spec', str(SPEC_PATH),
        '--work-dir', str(WORK_DIR),
    ]
    parent_ram(f'before {task}')
    started = time.perf_counter()
    with open(log_path, 'w', encoding='utf-8') as log_handle:
        completed = subprocess.run(
            command,
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            check=False,
            env=os.environ.copy(),
        )
    parent_ram(f'after {task}')
    elapsed = time.perf_counter() - started
    lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
    print(f'--- {task} log tail ---')
    print('\n'.join(lines[-40:]))
    if completed.returncode != 0:
        raise RuntimeError(
            f'{task} failed with exit code {completed.returncode}. '
            f'Full log: {log_path}'
        )
    print(f'{task} completed in {elapsed:.1f}s')

In [4]:
# 4. Adapter/config preflight — no LLM weights are loaded.
run_worker('preflight')
assert (ARTIFACT_DIR / 'preflight.json').exists()

Preflight: PASS
GPU: Tesla T4
Transformers: 5.14.0
Adapter SHA-256: b8555210e80d206c3a7e620433d09ad6e2509467609acd4789c482f4875097c8
Dimensions: Model A 1536 → Model B 1024; latent tokens: 32


In [5]:
# 5. Select held-out functions and encode original + renamed packets with Model A FP16.
# This worker exits before Model B is loaded.
run_worker('prepare')
assert (ARTIFACT_DIR / 'prepared.pt').exists()

Preparation: PASS
Selected samples: 12
Original packet shape: [12, 32, 1024]
Renamed packet shape: [96, 32, 1024]


In [6]:
# 6. Run all causal generation controls with Model B FP32.
# Expect this to be the longest cell.
run_worker('evaluate')
assert (ARTIFACT_DIR / 'generations.jsonl').exists()

Causal generation controls: PASS
Samples completed: 12/12
Generation rows: 336


In [7]:
# 7. Score exact facts, semantic similarity, and identifier accessibility; export ZIP.

import ast
import hashlib
import importlib.metadata
import json
import math
import re
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer
from sklearn.svm import LinearSVC

spec = json.loads(SPEC_PATH.read_text(encoding='utf-8'))
prepared = torch.load(
    ARTIFACT_DIR / 'prepared.pt', map_location='cpu', weights_only=False
)
samples = prepared['samples']
generations = [
    json.loads(line)
    for line in (ARTIFACT_DIR / 'generations.jsonl').read_text(
        encoding='utf-8'
    ).splitlines()
    if line.strip()
]
raw_df = pd.DataFrame(generations)

# ---------- Exact-fact scoring ----------
NUMBER_WORDS = {
    'zero': 0, 'one': 1, 'two': 2, 'three': 3, 'four': 4,
    'five': 5, 'six': 6, 'seven': 7, 'eight': 8, 'nine': 9,
    'ten': 10, 'eleven': 11, 'twelve': 12,
}


def numbers_in(text):
    low = text.lower()
    values = {int(value) for value in re.findall(r'\b\d+\b', low)}
    for word, value in NUMBER_WORDS.items():
        if re.search(r'\b' + re.escape(word) + r'\b', low):
            values.add(value)
    return values


def score_exact(row):
    facts = samples[int(row.sample_id)]['facts']
    answer = str(row.answer)
    if row.question_kind == 'param_count':
        return int(facts['n_params'] in numbers_in(answer))
    if row.question_kind == 'function_name':
        return int(
            re.search(r'\b' + re.escape(facts['name']) + r'\b', answer) is not None
        )
    if row.question_kind == 'returns_value':
        low = answer.lower()
        says_yes = bool(re.search(r'\byes\b', low))
        says_no = bool(re.search(r'\bno\b', low)) and not says_yes
        return int((says_yes or says_no) and says_yes == bool(facts['has_return']))
    return np.nan


exact_df = raw_df[raw_df.question_kind != 'semantic'].copy()
exact_df['correct'] = exact_df.apply(score_exact, axis=1)
exact_summary = (
    exact_df.groupby(['condition', 'question_kind'], as_index=False)
    .agg(
        accuracy=('correct', 'mean'),
        n=('correct', 'size'),
        median_latency_ms=('latency_ms', 'median'),
        median_prefix=('prefix_length', 'median'),
    )
)
exact_overall = (
    exact_df.groupby('condition', as_index=False)
    .agg(accuracy=('correct', 'mean'), n=('correct', 'size'))
)

# ---------- Semantic similarity using the same MiniLM family as the author ----------
from transformers import AutoModel, AutoTokenizer

semantic_df = raw_df[raw_df.question_kind == 'semantic'].copy()
semantic_df['reference'] = semantic_df.sample_id.map(
    lambda index: samples[int(index)]['docstring']
)

st_tokenizer = AutoTokenizer.from_pretrained(
    'sentence-transformers/all-MiniLM-L6-v2'
)
st_model = AutoModel.from_pretrained(
    'sentence-transformers/all-MiniLM-L6-v2'
).cpu().eval()


@torch.inference_mode()
def encode_texts(texts, batch_size=32):
    chunks = []
    for start in range(0, len(texts), batch_size):
        batch = [text if str(text).strip() else 'empty' for text in texts[start:start + batch_size]]
        encoded = st_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors='pt',
        )
        hidden = st_model(**encoded).last_hidden_state
        mask = encoded['attention_mask'].unsqueeze(-1).float()
        embeddings = (hidden * mask).sum(1) / mask.sum(1).clamp_min(1e-9)
        chunks.append(F.normalize(embeddings, p=2, dim=1).cpu())
    return torch.cat(chunks)

pred_embeddings = encode_texts(semantic_df.answer.astype(str).tolist())
ref_embeddings = encode_texts(semantic_df.reference.astype(str).tolist())
semantic_df['semsim'] = (pred_embeddings * ref_embeddings).sum(1).numpy()
semantic_summary = (
    semantic_df.groupby('condition', as_index=False)
    .agg(
        semsim=('semsim', 'mean'),
        n=('semsim', 'size'),
        median_latency_ms=('latency_ms', 'median'),
    )
)
del st_model

# ---------- Paired bootstrap comparisons ----------
def paired_bootstrap(frame, condition_a, condition_b, value_column, n_boot=4000, seed=42):
    a = frame[frame.condition == condition_a].sort_values(
        ['sample_id', 'question_kind']
    )
    b = frame[frame.condition == condition_b].sort_values(
        ['sample_id', 'question_kind']
    )
    if len(a) != len(b):
        raise ValueError((condition_a, condition_b, len(a), len(b)))
    difference = a[value_column].to_numpy(float) - b[value_column].to_numpy(float)
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, len(difference), size=(n_boot, len(difference)))
    means = difference[indices].mean(axis=1)
    return {
        'condition_a': condition_a,
        'condition_b': condition_b,
        'metric': value_column,
        'mean_difference': float(difference.mean()),
        'ci_low': float(np.percentile(means, 2.5)),
        'ci_high': float(np.percentile(means, 97.5)),
    }

comparisons = []
for condition_a, condition_b in [
    ('hybrid_correct', 'sidecar_only'),
    ('hybrid_correct', 'other_latent_correct_sidecar'),
    ('hybrid_correct', 'mean_latent_correct_sidecar'),
    ('hybrid_correct', 'correct_latent_shuffled_sidecar'),
    ('pure_latent', 'other_latent_no_sidecar'),
]:
    comparisons.append(
        paired_bootstrap(exact_df, condition_a, condition_b, 'correct')
    )
    comparisons.append(
        paired_bootstrap(semantic_df, condition_a, condition_b, 'semsim')
    )
comparison_df = pd.DataFrame(comparisons)

# ---------- Full-packet identifier probe ----------
packets = prepared['packets'].float()
variant_packets = prepared['variant_packets'].float()
y = prepared['variant_labels'].numpy()
groups = prepared['variant_groups'].numpy()
class_count = len(spec['probe_nonce_names'])
chance = 1.0 / class_count
folds = GroupKFold(n_splits=min(4, len(np.unique(groups))))


def grouped_linear_accuracy(features):
    features = np.asarray(features, dtype=np.float32)
    predictions = np.empty_like(y)
    for train_indices, test_indices in folds.split(features, y, groups):
        classifier = make_pipeline(
            Normalizer(norm='l2'),
            LinearSVC(C=1.0, dual='auto', max_iter=8000, random_state=spec['seed']),
        )
        classifier.fit(features[train_indices], y[train_indices])
        predictions[test_indices] = classifier.predict(features[test_indices])
    return float(accuracy_score(y, predictions))

representations = {
    'full_packet': variant_packets.flatten(1).numpy(),
    'mean_pool': variant_packets.mean(1).numpy(),
}
probe_rows = []
for name, features in representations.items():
    observed = grouped_linear_accuracy(features)
    rng = np.random.default_rng(spec['seed'] + 101)
    null_scores = []
    original_y = y.copy()
    for _ in range(20):
        y[:] = rng.permutation(original_y)
        null_scores.append(grouped_linear_accuracy(features))
    y[:] = original_y
    probe_rows.append({
        'representation': name,
        'accuracy': observed,
        'chance': chance,
        'null_mean': float(np.mean(null_scores)),
        'null_p95': float(np.percentile(null_scores, 95)),
        'permutation_p': float((1 + np.sum(np.asarray(null_scores) >= observed)) / (1 + len(null_scores))),
    })

position_rows = []
for position in range(variant_packets.shape[1]):
    position_rows.append({
        'position': position,
        'accuracy': grouped_linear_accuracy(
            variant_packets[:, position, :].numpy()
        ),
        'chance': chance,
    })
position_df = pd.DataFrame(position_rows)
probe_df = pd.DataFrame(probe_rows)

# ---------- Packet-distance diagnostics ----------
base_count = len(samples)
class_count = len(spec['probe_nonce_names'])
variant_reshaped = variant_packets.reshape(
    base_count, class_count, variant_packets.shape[1], variant_packets.shape[2]
)
base_expanded = packets[:, None, :, :].expand_as(variant_reshaped)
flat_cos = F.cosine_similarity(
    base_expanded.flatten(2), variant_reshaped.flatten(2), dim=-1
)
mean_cos = F.cosine_similarity(
    base_expanded.mean(2), variant_reshaped.mean(2), dim=-1
)
position_cos = F.cosine_similarity(base_expanded, variant_reshaped, dim=-1)
unrelated = torch.roll(packets.flatten(1), shifts=1, dims=0)
unrelated_cos = F.cosine_similarity(packets.flatten(1), unrelated, dim=-1)
packet_distance_df = pd.DataFrame({
    'metric': [
        'original_vs_nonce_flat_cos',
        'original_vs_nonce_mean_pool_cos',
        'original_vs_nonce_position_cos',
        'original_vs_other_function_flat_cos',
    ],
    'mean': [
        float(flat_cos.mean()),
        float(mean_cos.mean()),
        float(position_cos.mean()),
        float(unrelated_cos.mean()),
    ],
    'median': [
        float(flat_cos.median()),
        float(mean_cos.median()),
        float(position_cos.median()),
        float(unrelated_cos.median()),
    ],
    'min': [
        float(flat_cos.min()),
        float(mean_cos.min()),
        float(position_cos.min()),
        float(unrelated_cos.min()),
    ],
    'max': [
        float(flat_cos.max()),
        float(mean_cos.max()),
        float(position_cos.max()),
        float(unrelated_cos.max()),
    ],
})

# ---------- Tables ----------
print('Exact facts by condition and question')
display(exact_summary.sort_values(['question_kind', 'condition']))
print('Exact facts overall')
display(exact_overall.sort_values('accuracy', ascending=False))
print('Semantic similarity')
display(semantic_summary.sort_values('semsim', ascending=False))
print('Selected paired comparisons')
display(comparison_df)
print('Identifier probe')
display(probe_df)
print('Best individual positions')
display(position_df.sort_values('accuracy', ascending=False).head(10))
print('Packet distances')
display(packet_distance_df)

# ---------- Plots, using Matplotlib defaults ----------
fig, axis = plt.subplots(figsize=(12, 6))
pivot = exact_summary.pivot(
    index='condition', columns='question_kind', values='accuracy'
)
pivot.plot(kind='bar', ax=axis)
axis.set_ylim(0, 1.05)
axis.set_ylabel('Exact accuracy')
axis.set_title('Causal sidecar/latent controls')
axis.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULT_DIR / 'exact_accuracy.png', dpi=160)
plt.show()

fig, axis = plt.subplots(figsize=(11, 5))
semantic_plot = semantic_summary.sort_values('semsim')
axis.bar(semantic_plot.condition, semantic_plot.semsim)
axis.set_ylabel('Mean MiniLM cosine to docstring')
axis.set_title('Semantic question under channel replacements')
axis.tick_params(axis='x', rotation=55)
axis.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULT_DIR / 'semantic_similarity.png', dpi=160)
plt.show()

fig, axis = plt.subplots(figsize=(10, 5))
axis.plot(position_df.position, position_df.accuracy, marker='o')
axis.axhline(chance, linestyle='--', label='chance')
axis.set_xlabel('Latent position')
axis.set_ylabel('Grouped linear-probe accuracy')
axis.set_title('Function-name accessibility by latent position')
axis.legend()
axis.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULT_DIR / 'identifier_position_probe.png', dpi=160)
plt.show()

# ---------- Export ----------
raw_df.to_csv(RESULT_DIR / 'raw_generations.csv', index=False)
exact_df.to_csv(RESULT_DIR / 'exact_per_example.csv', index=False)
exact_summary.to_csv(RESULT_DIR / 'exact_summary.csv', index=False)
exact_overall.to_csv(RESULT_DIR / 'exact_overall.csv', index=False)
semantic_df.to_csv(RESULT_DIR / 'semantic_per_example.csv', index=False)
semantic_summary.to_csv(RESULT_DIR / 'semantic_summary.csv', index=False)
comparison_df.to_csv(RESULT_DIR / 'paired_comparisons.csv', index=False)
probe_df.to_csv(RESULT_DIR / 'identifier_probe_summary.csv', index=False)
position_df.to_csv(RESULT_DIR / 'identifier_probe_positions.csv', index=False)
packet_distance_df.to_csv(RESULT_DIR / 'packet_distances.csv', index=False)
pd.DataFrame(samples).to_json(
    RESULT_DIR / 'samples.json', orient='records', indent=2, force_ascii=False
)

preflight = json.loads(
    (ARTIFACT_DIR / 'preflight.json').read_text(encoding='utf-8')
)
metadata = {
    'notebook_version': '1.0',
    'created_utc': pd.Timestamp.utcnow().isoformat(),
    'spec': spec,
    'dataset_revision': prepared['dataset_revision'],
    'preflight': preflight,
    'package_versions': {
        package: importlib.metadata.version(package)
        for package in (
            'transformers', 'datasets', 'accelerate',
            'huggingface_hub', 'scikit-learn', 'torch',
        )
    },
    'interpretation_boundary': [
        'Causal generation controls test receiver behavior.',
        'Linear probes test accessibility, not causal use.',
        'The sample is independent and deterministic, not the author exact original ordering.',
        'Docstring MiniLM cosine is a supporting semantic metric, not code correctness.',
    ],
}
(RESULT_DIR / 'metadata.json').write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False, default=str),
    encoding='utf-8',
)

for log_path in LOG_DIR.glob('*.log'):
    shutil.copy2(log_path, RESULT_DIR / log_path.name)

(RESULT_DIR / 'README.md').write_text(
    "# Qxern causal sidecar and identifier probe results\n\n"
    "Primary files:\n\n"
    "- `exact_summary.csv`\n"
    "- `semantic_summary.csv`\n"
    "- `paired_comparisons.csv`\n"
    "- `identifier_probe_summary.csv`\n"
    "- `identifier_probe_positions.csv`\n"
    "- `packet_distances.csv`\n"
    "- `raw_generations.csv`\n"
    "- `metadata.json`\n\n"
    "Interpret the causal generation controls separately from the linear probe. "
    "A probe can reveal decodable information without proving that Model B uses it.\n",
    encoding='utf-8',
)

zip_path = Path(shutil.make_archive(
    str(WORK_DIR / 'qxern_v6_causal_sidecar_identifier_probe_results'),
    'zip',
    root_dir=RESULT_DIR,
))


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

print('Results ZIP:', zip_path)
print('SHA-256:', sha256_file(zip_path))

try:
    from google.colab import files
    PUBLIC_RESULT_ZIPS.append(str(zip_path))
except Exception:
    print('Download manually from:', zip_path)

Exact facts by condition and question


                          condition  question_kind  accuracy   n  \
0   correct_latent_shuffled_sidecar  function_name  0.000000  12   
3                    hybrid_correct  function_name  1.000000  12   
6       mean_latent_correct_sidecar  function_name  0.916667  12   
9      other_latent_correct_sidecar  function_name  0.916667  12   
12          other_latent_no_sidecar  function_name  0.000000  12   
15                      pure_latent  function_name  0.000000  12   
18                     sidecar_only  function_name  1.000000  12   
1   correct_latent_shuffled_sidecar    param_count  0.250000  12   
4                    hybrid_correct    param_count  0.750000  12   
7       mean_latent_correct_sidecar    param_count  0.916667  12   
10     other_latent_correct_sidecar    param_count  0.833333  12   
13          other_latent_no_sidecar    param_count  0.250000  12   
16                      pure_latent    param_count  0.250000  12   
19                     sidecar_only    param_cou

Exact facts overall


                         condition  accuracy   n
2      mean_latent_correct_sidecar  0.861111  36
1                   hybrid_correct  0.833333  36
3     other_latent_correct_sidecar  0.833333  36
6                     sidecar_only  0.833333  36
0  correct_latent_shuffled_sidecar  0.333333  36
4          other_latent_no_sidecar  0.333333  36
5                      pure_latent  0.333333  36

Semantic similarity


...

Best individual positions


    position  accuracy  chance
31        31  0.593750   0.125
27        27  0.552083   0.125
30        30  0.500000   0.125
23        23  0.500000   0.125
28        28  0.489583   0.125
26        26  0.489583   0.125
24        24  0.489583   0.125
3          3  0.479167   0.125
25        25  0.479167   0.125
11        11  0.479167   0.125

Packet distances


                                metric      mean    median       min       max
0           original_vs_nonce_flat_cos  0.986583  0.988882  0.973656  0.995012
1      original_vs_nonce_mean_pool_cos  0.987253  0.988633  0.976579  0.994980
2       original_vs_nonce_position_cos  0.986586  0.989421  0.934055  0.997114
3  original_vs_other_function_flat_cos  0.912594  0.907566  0.879615  0.960212

Results ZIP: /content/qxern_causal_probe/qxern_v6_causal_sidecar_identifier_probe_results.zip
SHA-256: e6f2e628fec01356ed2402db15f2178aed3b02e483aa05725cf9fbd655d46575


---
# Part II — Candidate-free identifier receiver response

This section asks whether changing only the encoded identifier changes the frozen receiver’s own likelihood for that identifier when no candidate list is shown. It separately reports raw selection and open-vocabulary generation.

In [8]:
# 2. Experiment specification and isolated worker

import base64, gzip
import json
from pathlib import Path

WORK_DIR = Path('/content/qxern_candidate_free_identifier_probe')
ARTIFACT_DIR = WORK_DIR / 'artifacts'
LOG_DIR = WORK_DIR / 'logs'
RESULT_DIR = WORK_DIR / 'results'
for directory in (WORK_DIR, ARTIFACT_DIR, LOG_DIR, RESULT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SPEC_PATH = WORK_DIR / 'spec.json'
WORKER_PATH = WORK_DIR / 'worker.py'
SPEC_PATH.write_bytes(gzip.decompress(base64.b64decode('H4sIAMRScWoC/81aD3PbthX/Kph6PUtXWab+y1qVLU3TNXdOm8XOpTvLJ0MkaLGmCAYgLWue7/Y19vX2SfYeAJKgSNmOm96q5GQSfHj/3w8PoO4aEU/YkvPrxQ0TMuBRY0oa3Y7TaJOGZMyD20EPrtfcY+GCLgIcafx9w6Ij/Op1hoev4JE47HaG3zUsQsFugoyf5/ddNhq7DusNBhPH7zmD0WQ5Yn5vcjwY9Xre6Jgtx/1lMX+5K6gPgpzO5DubxBbR811n1B8Nxt3h8ng09ruTkUMZPXb9/mQ8nkzGw+FksuyOcf4yjbyQwfSY41R6G6yDo0+3TESHN6NDj8Uh3x4GUTIoERey2NhxqN/z/PFy1J0Mu7TfHfR8NIE5Y3fJJszp9v2RkkU9GidMLPwgZBFdMyVQj0ktc5GR9HsJv+7EiT1NrmhvOMJJy8lwOOx1HTZxvJ4zcvt0zEbgyH7fc46pN2K9oQPOHI/gzvUG48mxO5j0/AFY7hyP3YlyHL1duOC8BQhikQS2wBzGIx65bBFzHi5MzHtOb+RMnP7Ow+CfaMB4NMnH0SbgmUYJPCiGZcLWC7D4KlmZ8VjwdZwsYDwOacJQ+PldQ0eZ3VI3gYizKAn8gAnUNWG3yLLxIxOMBJJQgpoTCJpgEghpAtGAsTXeS+YRCBgnyDpKyA1zEy7kdB59++23Jy/PXv90dvrixYt59HFFE+SWrBhRUsm7bbICRn4auYpjoQVhEYpEzoq+LPov5D0kypZsgmRFeARXSFPM7jTu2yQzUDCX32AWGCnKbbaVL6t6yEIg6LDU/Mv2kSUL+aZq5XstjgSJNFbmbFFwh/ycJnGaZGqDS9Rw4/7CJEnENkWOdDFFACISmQgaLwLIS+UB9cyBD0aXiXWqHVMmGOrnkkLYs6CjVxUsMJ+4K+ZeL+KARlzwMGxSIVpgEIHPvKH+qev3LElFRM5ECj72lSuAkm51ZsiERh4VHlF8DpERWdNEBLdgKtCKTSBZW3MSmtMPNJTgiPcUnpCzbcxeC8FFxjuIlHuWv4KfUQSAJIiJ0nW81XI786iqJExGQsCJCDVyGVrTJlHcQe1gVmaZ0qMsuTlvXALNJVmnMoG4Eu5bE0kCdPNGqySnCc8DKdOlh09RVkddKYlLrNdWIQ4/YN9DU8C8JROtkpKWs3LZOCvygjX504z0nkIMIBaz8+4FTuj2Jg9OEUWgsT487kLWBdEV5ssfLglQw2wVkTRMu/3+YDA8ird5PiOFDwWImX/XyBaAnaRHopgKulb10QDeDSzEaJEPduF2ReVC6w0DCfgAhaPmelaufuPiHgsOkMil4qEyI/8iSgDDtQhkwL3mj968AXPYlGyZxGElZlq4CDUuLyTd3rHCO7u209gDuGpKFvptQOowZBr8Aq9NkiAJ2ewnHoEZHpOuCGJ8ZkZiEdzA1JkKkJ2Qdrnh54MSQWgEOBfIBBLFEqTgMbrCQVgaFHzyqyvmwdpOUgkQXfA5Wym8/ZQGgLiK9GAjAMgWBTd5QKTLY5aXPn6myoNl48i5TtkLcBgwKp7999//keTN95BwWpDXqTBSbtlhoMe4TsqCW4c0cz6tCiPLpw/rYxECx5/VBQ1rOJqYkHOEFkYjYPdxxbCw0Ltreg2aohct/5spFt8/gzyfpiGsSCq0tgd08kEuv8o5GJU/RBKaBrkixZPO/pxIRUhmMHpkxe7oazlvkK/LgSqmQBJRmHNXxst5Q7l+jvWGF+3dx5bvkMi6rZAaVyBZdplT3NuAKNE5M4JF01kAADXBnLZScIZfrQp4Fj55i31xB6IlWVMzalUw9P9RMDZMbul1GkN3ex0GgJPY8hymJrr1WKlBZAci0Tsag6xwqn4K46SMLmKh5mqf7+Lq8FFcrQPTR3GtBGk5mu0C7nAP4NaA63hQAVcuFhuovUyNa7ZtE3A2dF4ceg7FrQqcb7FKs5nEDWmK665Vg3/VtQ7cqoMZ9+oTJc0exp6iVqUp0alYFmoyGS3ZX9awKsOGoqkL41PKRMBki7wgzrRca5rATYWATlkRbhdB5LFb8g20HwWtpoOdChdMkzXv5g20HFysdJ83cpNhqLBk3tAGw6C6uK8WZZagpdqr836phejKbrKFsjj8VfLoU309ZPP3VAQojn8yZfFa6bib+YNnZf5Tkm4nywdPz3LHqWS53NC4qSnasMbceoBZs155h/COCZ+LNbaAQG30QTSD/zAGGAUbOb2AGkbsdgW+T4IbBvsfnBQj5FFoGDUFVHgGgGs8HUjXRMbMxX2dR1AFbO47GgN175kuJUtQBDKxNkOyY6cxhY2qSDI7yAvon/UDI3ZGQhg39hbNfk6vCsA8tbI+ez6zn+vHYd2gjz0u7moFja5Y02mH5JB0bY5IYVmBtLZRht15MA2+McIvWjtlCHrbHKD1T1LYAj5lKn5gB5o7xUyZBhcP0UF1WwIfocy0yHSY1rDeBiz0imklJMpdPtsTE/wsBaPXFRT4Y6erjUex4FuXRwBIURjv2cqgOjtQZApaHyWg+F3s6e1ij+oFHwWfOizYRZteHdpEvAZsht0K2EAnw8MbBquFq85xDMpltwuEu2qa5J/qNsZjCQ1Cqe6qizF5hQuANGdLPsAqhMnLxWFI8exHZHp5uMVNmNWEnbLIAzPNc8JucBoAtD6vgvAtQ7bWBzwuyvLMvjZIAhoWbCA1ocsTBGICHSDM9oPb0uYkw7DaZa7jQppD46ekN0uNz7xhVJs3HnJc4UDlrrLHqy3sH9xtdgV5gsdLfgsVBIRemmzri2g38fas7bZbanpc474vs9I/pRbKXa4R//z1f1RtcnHTIY1o1aJBM02v2KzbRtRa6GvnoQOCvzE8tJGQOrCHRnpEPWRrtlpSrXVUc6/b2qsH1hb6lFHhrggYt5bV/buScB5A/kHAgfod3uuDNUxMcAS0rTe1O2LApJpNt7HS5vmTZgd2wH5rLZFGyd3D1GnV7bG9wE2A1116kPCEhgdT4rSJuVESZTZk/AS35x/Abxf3T9h5S+WkIxU+m8Dss3XrrYn0/tYOLn5Z8c0uWmU25wpcULN54wJ4om7WBnihOikk7FyxpGkRt6rdOpJVYObzMue37XHRT3uKXrHXz65002/88WXqfE+FFf7/Df38qLrEQiyAQG/NoB3k1INArYMCXnwONqxU9B4r6zTmyAB7HYBoxSY7JZNpHIfY/RhZsq6yS8KgsHWkIaYqdVqqwAwJkw+cN5WSGlYPl0m5KDFvlu2qX0YXMQfKA5ev1zTyjsBTZ0b5D/EJGndgHwPVputnecTO2JvutTq3Vrn66bsgMVT16boviHuXLcv4R3vBJ2XtZ6XRU1rE2gQe9yr5C1xpFHHdLWtpr6CuZRv66T1t3vtMFm6jFIbDX4Bfqg7fNuYIVaawqFCkyrjrAzrohiAbcMsheZus+AY636i8UJ0yRqZg3Wp6+TIM+eYM9X+Z8+kULOXlpQEt+3jYZEjho85uhmdpmq6bpKs4LHBLqNIWvpibNHMvtPH1TNssP4vgKsLjFUuFVmvPe50v7KHP3ciUIrsnk5WR+pcRlUTuPyuRa/KJaFeCdyup2396L9WvpC6ETb2Yp6K2TMzxzQzD9xDy/oBsiGJDdOx/J5idWmeL05JYNUSa+B6i9bvAsmobzCnv3YESB+0PHsp1vHQdy6Y+6bp/GpJDIJX2p6j8wyD+sHOfj9hW7J8E0vtODZ+X5E/LvOcn+2RUSXb1LqLJl7+2Cf7yJqbJqu5nBR+RTKGfec0LPTolb998/0ZNy9P5Xa6Xvj/MP+ZEzcgg6nzbfkuCuArjwHeTy6ryf/hXBMqMtwCnQSKoe/3wrwhOYaHnwvy249J6Fd0pOFzi4bOUxmiZ/7AA7jvac7nTKin6mMue+05cyd1JTpCgiI0yzzhHeuS1eF2aPO1E6WlvxWs6CGxWTBVkvwmrwu0JdnEUQO8keEl+eXtSzhb8vBRX6RrqfefIMWNJmhCxln57imMYJZS8Hy6/grw7+eW1WksViuGv7PDYFjdVnV/evTz70ZL1lfracHFNBU9hCjYFt+twGnj5AUrTCySFK6+1O88HXOYxi5q5B3Yp8NjX7wgGvmrBH9g7QR0cGAkzANEDgn+rnDtuyCVr7j5QFiUCOqUZeQ3KgefO4M68LK1Tg0EqTXfeFhkGoAVe+YKvcWbddEWumANxM5+Me1HBOSx7rdJx8uMOVzphJqDa+tcoG6hAyUPIU9JU1R5B1w/rhTrJAtl4WgWjGHZmAqSwKHt3VvPazBiIjWyl8Otz8nPbLNRmzxKU/0TyyxwX1xXa80+Lq2dTCY+bUZtsAi9ZzT6++f7sR+jZkm3IZqdn/zh5vfOOCryY6DNLmEcEBs8HmQkWiF2Kpo9YiRQSK2Pfhl4pYhbz03N1c9EBZlWIflCYHbAoEOn6SA3HOKc+ZMBjJ2KqP1aaqWYYVfkynYLtU+PN57cFA+f+4v5/70Y9ouEsAAA=')))
WORKER_PATH.write_bytes(gzip.decompress(base64.b64decode('H4sIAMRScWoC/+09a2/cRpLf51fwcliQ41CMJGdzi0HGOK/j3B7WcZzEucNiLBCcYc8MIw7J8KFHFP33q6p+VfMxkpwEyIcTEomP6u7q6qrqejU929blwYvjbdd2tYhjLztUZd16SVGUbdJmZdHMZvpZvauSuhHmvmn15T5p9nm21rc/NWWhr8tGX9VJkZYHc2f6abODmBEeVdJiNxqJd3BrRq+ars1y06asN3vnJioKwMgrzMAtjNdsy/og6kZ2v+92u6zYbZONiPedGWa/xbs4La+LvExSCctba8CXXVu+Kotttgvp+psyFfnXZf0q6Zokf/ONfPq+vBRF9ouoZ7NZKrZeUyRVsy/bIE/WIl94TVvPvZMX3tuyEIuZBz9XB2+p5hddZXXbJXl8EIeyvg3mBFDV5caCvIM70TRB2UQ70VZZGswVWHKLEwDIO7rHH59G9Rce/Q3t8+a2acUhTq6SDF7lIt6tAerqEJkn3mfe2en558+ePWfNKjl4XMP/1AIfRBLbOCu2ZTCP4N1o202XJnGS5+UmaUUqmwfmNf7IlUQ43aUBD+a2T6dNtuXNssZOCZqIvBFEadNk3keoFo2orx6Lj4b+ndG5V+ucFW2A0hOl3aFqArWi89Db5l2zX76vOzHXfLVPzv/6RbzNoGcUHOIs71eSGmIwuJX8lWY70bTAFkpMI9lUMdd11u69shIF9RJ6fr325yhKe5DXXLEo/oA0eJt9V1x6WeFlraiBow/rNFkoyKgWSRogTbxnRBpAe+3784VDH4lM1FUprGpA/Uk8agEqqNDv9+JGXgU4302eAE+9gRZF+93XJJVBUUQggF0u1ABIkxhYMGvj2K5iI/KtXfE0OwDEAibQug/Lru09LbpD3KIoN31wUSGxB13sszQV/b6322L8BTwGasm+YWX+xvqqy4rQ2cLS47vT6Ey+ZZRsugpIMI/MhOczxoAWHe8vZiRnFdRD6L0QN63L9Pizp9Xe40oHZ1+EgJ/3eeidh97ZfDDA3lsCkk4Xc4f+0c+dqDNBoxXRu6RODgLZZ0TUcI8oAkv7kA01B7Y6jU7PZxOjgCb6CcghR3mTFSKpA7ngTi9uozWol0uFmWSnNxkxHef6GOkAqO1EQKvfY2nWUZRUIEmpZc6vsk0b3A3o62/qsmlA3yBgl7cZLsfLFvkb9tzherhMFur1CzW3LNVfkLik3exBK9RNS+pi0BVTgAabAkTqTGLzJrkV9Vu4DxjNRprgpP9U+J8/DX+QTNngBwHsCWgn+TjaLjPpCVjBHulcNfuv129+DKZffyWnHKipTwOq4e2QDj+Hj6bQ80dT6H4+JluA5EC4OEI4CbcZjsqaOQMTtNHcIGbXSZ32FLe3jzdgZC2UdngPGqGsQy9p2yI+JM2l+0IqSdz6+FMrq3ILB1y4ugjkEBFp24DNG5WWAVYaLOqKBi7FLyI4nUfipgJ1pTtoMnoaeidn+L/tqErSFMxOQhj6M8i7ahP1DOkQ1DVMo7iqJkERS0Uagk5aSviV0iUXQ/alKYRq3vbvpbiNOVJLfjOhxzlF1LBSaVwE8vGnBrf5wxiT8lAtQ4//eXDE8wdHHGn0nDVSj1H+1UO26MoIcZg3cCTANJFWGJpncZImldnQmkpsFiAPG1BmtIlpwwztPhAUcZVtDEfLu5naUFQ/MRpiaK65jknAsKzKOEuXONTKX3dofcX40L9gpgcYhgXstApK960fc9AasGhAb/c7lE85JE1oSb+VTSIR35DTAnYlCgwzTPmMJChYDwz635aei55s7F9Ytq+TDIzm77sC/cTXdV32DAf/pWwKtqkAeekO3iFrDriLLDzfgdx+cmeHvsex78YHv/+kZ2O09S2zvcAxFjBPuYLuyvRXEYQuqWJyYZC+/qbq/NC7Ftlu3zZxWeS37iYnhxM3GzAzvPe3lZzw0bEfGs6Q3WIJy1pAT8VGBNRhSOxqpQC0Gmzw+CbGFz6pJLx1IPq9rHiTC97nfDiBkRZyM9jkIikEObHSKULFCBor9K6SvBMGlQg8kEMTsK6pJbQDYEerVrXYZjdkyvoHMsoiWAO96HT9s/T0o76zAlSjXiMYsm4bdJUC2V0PkI9Pf1c5OlQK9MLFUaQr+nsBwDSpGaeMApkpp+jnLquJHHb5fLUbIeZqF4skS5knoKjMI9tO2ainESi/6LkLIFcKRAdWdQfDrYCMmvZk/mpMgCb4qChbsxYXmsVU877w/lPcSsHdusJalej96DGhV/CH7tTtva+sg3FnwLKQJsYFurWVmGlTE1URutMKrEcoBb06u5ibFuuRFoyQusmpakLUBDWM3sPqgvOqIQyzPVAhAWAtIroMarUaH6LgQ/rp/APyITR1/DipxRxGM4NqTwMRJrhoV5ddFZzN1XZGngqMeEhuAtNqDhvg2cz1TN1ZOzxyOjZ3vlVBW9cr77naS1qJga9Nj9djzvaSLbfrci/p95jPvWSG6IjnvWSGs2T1qC0DufMCS7WgZZdSn5IN+PxcxyMacGzQYFOKgrZ6q7C06oTNHe5UaIaRJhIg28qRJDWk/V7kDg1jHjqazDyNlNQ18a6GwYOvk7wRTrhEdSRFI5F/1iHbYkM1EWWvwE2ZX4m4Bbcf7FYMZgbyjxp/J1q5rnAB9lWt3oJigQe8GfArWjRme9kkuQxzyR7mbuDoEqz7XUNK+M5PBVrMNbgjSLX7EFR9T586G67+kRvAUqEYPHsmO50PAAEbtVk0pKdsqLX/o6hotbD+md6C7SI1jVRQomlJQw8oNkUtNapqCNjqK4uuDBXKtmrpJAvH5GeoJUNfB2RXIsdGI0UzudCzHjEJaQYHmLOx+utsKOtSNU1a1LiyBxAwKSN2bfR4CIdx68AZw1F7EyPwXgLbDSfy3ANWA2aaGmC6c7lRvQRKZOuuHbMzt/4ryopoyqr18LB7j3IGd6hKNJGjOEYzO47v/Z51p1YfdS4ho12JnShEjdqlKMGmiquyzMH1EKmMHAJlwQRW1xS6BwtjpyKRxAh51rQr0EbKdq5pG5dJl+h7+kPdSSQAy6YskqLF3ctfb9Ltbv/TZX4oqp/rpr26vrn9ReJ9VV4DKyJQIrKykw9xZmzbg25pQxZ6wa/3YPx7aAIR5Nz7UqFvDaF9UrMetJrIwJ27sdE2Ns+egkjyap+sRStZTk8F1lf28BfvnNxrKUZyDk57Gl/voUCqaLMvYT8IdL/MJ8Qp4Px//uUm9mH/9P3oJzBcAurCYVyC1IYR0gTNZHwYga2cYpBpm2HYth8+FEUEHjjRylVmRD0TUzSvtf7AtzobIHKxaeXOGZM9IFJiwCY46pAqXFqdtCKzyUljRcjbMZiyLcgIGKYsREPO0wGzYHEC1gUa/T1XUr98yJcEhd81LYAdShAA3B1oQ+WmqUVx/XEoro+huP49UESRJZU6FOQeRvZNjDKJaA1foN5iyHAAJhgaRKJARiB3nxTZyCF0nSriVuBUHMxyJBiJxAOMI4IBT4KBkYItBAhlYGUoy42ME6vroqyouhaNTqfvtdP3+vfreyslj9DH2ambtStsoCzarGCbPVj5GKnAPQX1FbWfh56+XqtrEj4rnIrMEag8kL4E7KtAdRSCSpsPRba/FCt8deGkaslF0wIj5415W4vUGOR6CLkeh1R9ItEAmLo81iODY4a65B6RZ7tsTURbmVeWAErnS9+bP0Nu05RT7jtfP7ZfvFg67E47Ke0gvtwvjMeJy6zRORoz8t+WzgKcKCXp0QgnJAyEG+DdFakyGP7dewc+PNjDW3ENv9uyTXJPe6PtHlW8l4MRCu9UZpjEKXLIFDVl3Qbg3C1lwhSTpweeasZ7cK3gP9he9M0ZE3304SxQyGDYWp8QA+Ab57GCVUrCFBTIZdnADpVhJhb3Yo0vQEsbgb/EOYBmtc/m01bGuIpD1/M/TCtwKLvtFtyFfo9yJyMVZl+tFlPscDFz47bUNrQzvHP28cWYEHJlqHuQWXkp+xxDudtWdXZI6tt42xUbSni1tYC+k6aNXv7w3uypFajoK5E7YlKAkMkR8QIFQhvg0AVY3usyvfVJiXDRYLE2bBd6AY71tRr+K7EN5eDNbbFhT9VyOzQyaCG7Kakzz+ZoNZ2xOgWde9edfk/B45qGfwuYvLd1MmN5eJnFKXNpi5DhAbpAXNtbHhvEOLuGBaLpy15OSbWm3LW8tGkk3MLbmFFAYYBEYyOR4khFJPtZuiP3MuUWzh3fxYr2/GwTEwK0RoOkAj6UmEos+4uF0cA++v2ZvYWBj08pSx8zIYSamI6DrYoZkMAZZkfbh/zK0fV0Sk+QqTGEAtxCtWO2rQqMoE0yKk3GYQOQCW/tf9B7Utr99U0ltYa4STZtfuuhDw1sfSIlUHet1Dpjsm0RmekrbPu8rqHtTOeRXGeLKM5wm93EKnppYvANg9EhGyx/KSQ55MvZ7D9lACorYK8RqOJwL4bdEckP90i0KtlciraRuwZt1aFrF4c8+hQqLy8VzcJ6hvIp7CbD0hYVD5ORTcrBo/2p61M+D3WOiip2eiFPio5br+1UqU0cfB6y3hi7rlU8lIBW1MFCdvMpa2CdQzL4Gm47upYjtXHtGUluYDbMAjdLv+KxcJaYXfpIEGVMh70wFLCCzKVYW59tyqrV0lI0ZEaqiTJaVVB2bUXpcFpAdwrPnslJuqPIFrGOZWDMsRlBpmtgR0zAoJGm8iglUh2nDEdKaEwsWA4YOQOuTs4udIacRSmJIW2ENNDFAHIeKz/RBSGUWfbZlia5SJvJqqdoU3XBPNon+ZYn4gdVcT2TWcpHCsbqAbxyzUTIgcQUc86NfdOYDDplOyt+tbZur+zO2UOpwg/IISdCcdflqTUPxDbHuPmjMsJqhWUN3zvK4mEL9mBmtlbkJpP+k7cBlTEuTpVqM+WmvsHihGjiO9HpQcDYpkBDmdQ28e9hiltPSk1HJ7WZGyqDYCaYIKtm/yyRBIXc+iOQ++NjCNlYoBe8P0xXy7QWFuYNIdYGYv343PnW/wpeFIgyy5q/9O5GUbj/7I4wALO4l1j3/z7aZK2brAdB0GzLDF9bag09f3ctiufxX1ldtY5AT9kB/03l2WOlAcO+sI+usDXOsGm1e3j2ntd7r7ssT0cwRi0hF7xJDlWOSUei+tn5o0wT8Be99lpg7Fgc1gLIlaJyK+sT8FbWwtOdzm2MNXQ96Jh7L6F2OaUr5KSIHwgHKsGdM76T8yJEYutlwQzRzqZAYz8GJTtX+PluLGMYx1BwxhMdj2FoKO78+hhoBaMOtJPapC3cOYfT/mvs0AWgnXtecD2YjiKtmbXz1DQh1Rxd1+Baky1r6c52KUlnDEcX7fI8BDOuwSMWSbPJMrlJz0Np25EN0rXbk7/5PJg4Vs2P23GHhPLLS57Z71WwLBx9bsFIfP2F18vK0vO1es6iPT4mY3PK79olws1Ypy91WZxJDo+gJBPKao/VDeXDMXC29/i0TwbsCW/A1UZ8Bb9w3RbOwY0o1i/imLXcVZ2/4FX6mNmUWxgxPVbxMe4rD7A9o3VVJessz9pbX1rUwXgPFg764R1pmY+1nC9GFQo/o6GCEY5ALpRmYKs0Io6LY6I63lbz/mJE1TBy6GBIjDQgfxdFri1hiTYiu6K0bs8GRbVyqFoVUlxVKx93URkgpKCL1j0I1AqgBBqd/sUFl1VmJj0kekp0rPQ9IGlPOXYBFgK4biJ4qlH3sSYcDuYYcMYB+tMnasDlTyUbcb8tEmUjnz7aHo0/0gpVk1KE6h/XehK9pmolH0u3GXMpJ+mHP3l5HYMfhAfAYsBz13+vtMwhqZZ3PkjT6T17169lOfvCvqS64wy1DFh8rSoQbNIq0QVgvGCFEqltoGZH6k2mXKQSAzlqwFOTxUFK+4OLaI3U46F4WgkwM6+ztN17aSlk3YOskdKVRTTepEQQZifJCbKFSE2tmMpLTxozqvShzpKiJeqzSIZ+fBBUycDiG+sEvOsM9InU0qiyBGyNlOQb6G+37oVUKTa1aLjNZcKjX+XAEDRJ6l4UTo648hEIBdoOMJ+P9obz0p2NnEhRkwSW0tMdOUUgZ6O3iHEYgweCmZsRSKpYXnh6Iv0CZgNX1mDXFUmuO9UNtmAUAsVXfr+eWXrvkiW2sJfENlQxFkxjqiJk4TSrmpz16CdkMfYjo5rSQroInViaNuZwvJgkBYOODCldiLhwudCi7ACDHYTgFvFRE4Kl0kLvGR+c4UTLfBBJwcbqja5jPhGC4amN5ZmOz8ymjdReJ3is1H3CS1L7SGj+48/CYd/IzKxjvP1DzWGdTEfV2iRXwtoYbKefMCe4/Q5NU2XX8obhFPXkqmmLs7889HJ+hJxu8wFhnQ7uB/bOD//64f3rb+J333/7zbv3xr/85F9l54EO9hJvA3+2XU5RXC9pmgxzVG3kfV3msI+Rw9uB6+U3OvCJhnnS6iB99MlsPnvz8v3rt+/jd29evnr9j2/ffPX6eyy2+fLLL+WLH168eOHbREQKiwj+IBrfRk6BzXGj1FrUqR0cFBKaCJ42R0Aj5rfUpzE9hydp9BBD7aQ7Gou7SvsmjVVBCPoh0sgdCeBSwrigk9jtPisu0VQd6dKq9uMVin/WaToHHF7TH4Bf9LNPvTiOqrgajLr95MtfQUbJPP71xZ1CH7aSEt2c+w/FJyNNLBgWgwhMIt9TN8Bdv74YbQMbuWqEm7em0pGjUp8CBzPMjGh8KHhYSXK1KbfgPM3SayqHsJiNraulCbYIR5MexwtqWDENz1jYSHZ2w60+ln7ie6VUKP2Desal601GB7SzLQeJKD4TDBWCDLIdi7G9o168A9jWVOKToI3FMoEyiOEB22/Evsyx+HhuimapWnZUs9i0/Z3kqYX+cAKetNDss/AcRckMctuK1KDThk1ctbjQ+4d0bYF8ZSMB0H9CAjVVno0SSEqVXiWdX5qw22fGV0CDSL/Q1jy9MIjI4w2jLGrx5FkuwvlYKz0nt9kgsWJJbzAMFEJsw5Ns5x7LHBT302/WiHWokFUvL2Qe5+yhjGyzKWsM9OiykIdyslKKesLhlthIB2WQqtWWpE6+GnB17MT1Nfu0lklrRebIkmjgoaBW66HillklxS3WlkXotoCPKGtrqWA3bdzGSM3j3t9LC20IlcIVDAJLU7W3XkNHsje6atvmgzEOwAtyaLSZm4xVR17QBmMYz6exVYZsGlNRxEj0Al4YbSXBBomIkeYmzKGbY7pZpCb0A3YTSwAHdpIhmwyv44IJyWHm4x5/Xha7fpxgOXRF7OTVgWTZGlAum0fgMxzSHYqNJJv0agXq8jrU62C94Cn+MUvaX07ndLVIV9TtQoJj8J6XhrozHoCe9UYzHrea8uzpyhVcWUHfuFFnD5dKBegj45y4/Li4xZS6bLh+DuRMVd0MMI+FYVqzN3I46NLqOKuZXE4Axp5iBDULc3TuKdzgFgO4WDM0wt5yMZ3c+yIPlk/t6xKlTZGdrC0pYqBuokrUW5nzwaJ7FkRuhgUYsmRB0WjJyOsE0NgElu6tBZusxBivwpg/PC2RJ1UDS3pAvIORqXkneupz+vTO6akkVl7u0BJfN6aso4FF2mXMsUeQpty2qDKRzicuYzjy1199yerl9ZiEB3LvyfAbH0qOXIk3ZPklq4Le3hO6Glr30AxOE8P+nVGVlWGnRJYguXU+zlxOvLPwyOtPvSm4EY3qGv5S23Oim2updAy6RBhb3IQ0HI/M+S5lVOAtnILhteC7tlcP5TeYzSt3iJCvPi8U9HCOAAa4ggpxBmXkGDh4qD0FioYd3DtWHs43ZGwdumugfA86CB0nRXMt5PG0kfpCWX1WUhaJHZD+UHzXiYY8SzS6PxSv8CsieGkdPXOnfDSeTv1QoL0Ozho1/lHZ7l+SYy4bfqavp1hSm+zbDBYVUXTqeQ0UODYnZ72zj9bgXy00oFvCS75AW4PkPGSqmrMo6nzTlPs2ZqHSvo8lj7xYsOfBTWh1dw9xFMfi/OIp24aeQMr23aKnQlzlrXa+p+ptd7JL95bZVaXK3/bVO+bK1yI5NMuz0LELtRG4HLcN+Tag1tfh/P7xEWguz88GhjR0JqC5zKq+l28L17RDLzOnGJzMA15xhiKpvA/coHQaYzLbCmKhgjIs0ElKbqTOQW8LVCIKA1lR+JQ6Yq5ZL0sbOshjgqozK/9ABlZlilKnsA4kPcO0AezC9W+prdOYOJlZPeLEt0EchB7xYRAWopnIDOgeV4OQu1utORbpt22HIXjb+vfIOP/xNXNPyzjLgeWxhuXTq9DMty5YN48oTHu4Fs3nSeu1VndyhCdR+niuev0nzVU/P/89ctXrJ+Wqsd1APlcq/3PR39qnU9l/P57Klrb3tBKRyez1RyWz+1UzvEGvnEYOT7Er12pngXv3BaqGHPgt7rdSh+P+N6kPJ10VYkA/7zC6QQfhZbFWI0/B0af8VEKA9I6Llj5spsEcf3kQeGYC4tgvPf23Og29U+dbWBaHFX0xYDxNOwjsjY3FMQ11mSaPZfZtroc7+cJ2YL8Xomqm8Dy7jqS7LlSflNwK1QVXhuKy5sr93NzHlTU4jPGYBTu2cIPNZ6WwGikEOLZ+42kYl3+P+RwoNqNR3cdMgFGixxBHcGM2UQ/N4bdGRuRwupLDrbizmYaxao3HVn48pV7j42s2rKdK06SkzeYWFgs6YCs3NWO2nDRrdj/e5tkztFgHr3TpCF+pFg/5Eg315eNKeWySYlhdsTICaHq/GGH6xwvYFI/K8cIJARp+P3T4aTMTYfh9BGkS0QdlaBw/Jkt9VMe/+/NYOXqqLD1FnmQVs155rEg2PPYgvJIqdnekzRPk9rfL7sfK78fK8BE5ZoGn3teh+N5KuaxxLpEeOP+iB4+iPJGpJ9vYMjIbb5ja3qZFYMSO+3/W/gNYW/IElpTRxaO47oETjE9ZCTqOkAvM8OAK2IXAryuMgNMBRwJ9+LzBaBEYfc6JBY2sj8ZCKd5nPAq9rYXWvhE1AtfZKlxmbn9Mvzv4ld6yWiMzxmxCDMLfOKI1Ud05zY5YZ+Gx4wW8HtC00KtjHvCyvt6MFGzvab8OsIeSajTyZrL+D2NuhyQrAhMNrBvSh/qfgole1rsOXfR39Eb/cyV0g5+0ihP1ngUOWzwKHHpSkzbLlT2nKr9eSk44Xmr3WDvik337JyfI1H5oPlLKAp8TDa7L+vIE1nu8EX2rcKnb0h9s3eiImPrwzOQ/9HE8ZvDqx69ejsd+cBowLvEKBgOaAEOUAQ4d4bs5/RsbMhTbD5WqnDhOjBh56dm2+qGeHeP3pW0BrK9fNf4AMjpcwu8AV6do1RF08PvxEFB5yWinw29lE4niKqthKvQ1vn98Hb//9p+v3/qWhoRbqz6Fzvhg4WQO2Enq0WMfffE1bUhQeyf6e/Tsp/FANEbQIo5c9ANFwUNYUDQpqnTgBz+qYjtxY9mPnZjp0n3r1BRm+P0V+e1BmkAcowzHsZqBFOjZ/wE+OyAY+mkAAA==')))
print('Specification and worker written.')


Candidate-free specification and worker: PASS


In [9]:
# 3. Worker launcher — logs stream to disk and child memory is released on exit.

import os
import psutil
import subprocess
import sys
import time


def memory_snapshot(label):
    vm = psutil.virtual_memory()
    payload = {
        'label': label,
        'available_gb': round(vm.available / 1024**3, 3),
        'used_percent': vm.percent,
    }
    print(payload, flush=True)
    return payload


def run_worker(task):
    log_path = LOG_DIR / f'{task}.log'
    command = [
        sys.executable,
        str(WORKER_PATH),
        task,
        '--spec', str(SPEC_PATH),
        '--work-dir', str(WORK_DIR),
    ]
    environment = os.environ.copy()
    environment.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    memory_snapshot(f'before-{task}')
    with log_path.open('w', encoding='utf-8') as log_handle:
        process = subprocess.Popen(
            command,
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            text=True,
            env=environment,
        )
        while process.poll() is None:
            time.sleep(15)
            print(f'{task}: still running', flush=True)
    memory_snapshot(f'after-{task}')
    tail = log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-40:]
    print(f'--- {task}.log tail ---')
    print('\n'.join(tail))
    if process.returncode != 0:
        raise RuntimeError(
            f'{task} failed with exit code {process.returncode}. '
            f'Inspect {log_path}.'
        )
    return log_path

In [10]:
# 4. Adapter/config preflight and deterministic token-matched name selection.

run_worker('preflight')
preflight = json.loads(
    (ARTIFACT_DIR / 'preflight.json').read_text(encoding='utf-8')
)
print('Selected identifiers:', preflight['selected_nonce_names'])
print('Matched token profile:', preflight['nonce_token_profile'])
preflight

Candidate-free preflight: PASS
Candidate list visible to receiver: false
Selected identifiers: 8
Matched token profile: 5 tokens in both tokenizers; 12 characters


In [11]:
# 5. Encode the same 12 × 8 name interventions with Model A FP16.
# The worker exits before Model B is loaded.

run_worker('prepare')
assert (ARTIFACT_DIR / 'prepared.pt').exists()
print('Prepared packet file:', ARTIFACT_DIR / 'prepared.pt')

Candidate-free packet preparation: PASS
Encoded variants: 96/96
Variant packet shape: [12, 8, 32, 1024]


In [12]:
# 6. Candidate-free scoring with Model B FP32.
# Two prompt paraphrases are scored; greedy open-vocabulary generation uses the first.

run_worker('evaluate')
required = [
    'candidate_free_scores.jsonl',
    'candidate_free_greedy_generations.jsonl',
    'candidate_free_base_mean_scores.jsonl',
]
for filename in required:
    assert (ARTIFACT_DIR / filename).exists(), filename
print('Candidate-free receiver outputs are complete.')

Candidate-free scoring: PASS
Score rows: 1536
Open-vocabulary generation rows: 96
Baseline score rows: 192


In [13]:
# 7. Analyze prompt-free receiver tracking and export an auto-downloaded ZIP.

import hashlib
import importlib.metadata
import json
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import binomtest

spec = json.loads(SPEC_PATH.read_text(encoding='utf-8'))
names = spec['probe_nonce_names']
chance = 1.0 / len(names)


def read_jsonl(path):
    return pd.DataFrame([
        json.loads(line)
        for line in path.read_text(encoding='utf-8').splitlines()
        if line.strip()
    ])


def normalize_identifier_answer(text):
    value = str(text).strip()
    value = value.replace('`', '').strip()
    if value.startswith('def '):
        value = value[4:]
    value = value.split('(')[0].strip()
    value = value.split()[0] if value.split() else ''
    return value.rstrip('.,;:')


scores = read_jsonl(ARTIFACT_DIR / 'candidate_free_scores.jsonl')
baseline_scores = read_jsonl(
    ARTIFACT_DIR / 'candidate_free_base_mean_scores.jsonl'
)
generations = read_jsonl(
    ARTIFACT_DIR / 'candidate_free_greedy_generations.jsonl'
)
generations['normalized_answer'] = generations.answer.map(
    normalize_identifier_answer
)
generations['exact'] = (
    generations.normalized_answer == generations.target_name
).astype(int)


def matrix_for(frame, prompt_id, base_id, score_kind):
    subset = frame[
        (frame.prompt_id == prompt_id) & (frame.base_id == base_id)
    ]
    matrix = np.empty((len(names), len(names)), dtype=float)
    for target_id in range(len(names)):
        group = subset[subset.target_id == target_id]
        mapping = dict(zip(group.candidate_name, group[score_kind]))
        matrix[target_id] = [mapping[name] for name in names]
    return matrix


def base_mean_vector(prompt_id, base_id, score_kind):
    subset = baseline_scores[
        (baseline_scores.prompt_id == prompt_id)
        & (baseline_scores.base_id == base_id)
    ]
    mapping = dict(zip(subset.candidate_name, subset[score_kind]))
    return np.array([mapping[name] for name in names], dtype=float)


def loo_center(matrix):
    centered = np.empty_like(matrix)
    for row in range(matrix.shape[0]):
        other_mean = np.delete(matrix, row, axis=0).mean(axis=0)
        centered[row] = matrix[row] - other_mean
    return centered


per_example_rows = []
self_lift_rows = []
spec_prompt_ids = [p['id'] for p in spec['prompt_templates']]
for prompt_id in spec_prompt_ids:
    for score_kind in ('sum_logprob', 'mean_logprob'):
        for base_id in sorted(scores.base_id.unique()):
            matrix = matrix_for(scores, prompt_id, base_id, score_kind)
            mean_vector = base_mean_vector(prompt_id, base_id, score_kind)
            loo = loo_center(matrix)
            mean_centered = matrix - mean_vector[None, :]
            for target_id, target_name in enumerate(names):
                raw_ranking = np.argsort(-matrix[target_id])
                loo_ranking = np.argsort(-loo[target_id])
                mean_ranking = np.argsort(-mean_centered[target_id])
                own = float(matrix[target_id, target_id])
                other = float(np.delete(matrix[:, target_id], target_id).mean())
                self_lift = own - other
                self_lift_rows.append({
                    'prompt_id': prompt_id,
                    'score_kind': score_kind,
                    'base_id': int(base_id),
                    'target_id': target_id,
                    'target_name': target_name,
                    'own_score': own,
                    'other_packet_mean_score': other,
                    'self_score_lift': self_lift,
                })
                per_example_rows.append({
                    'prompt_id': prompt_id,
                    'score_kind': score_kind,
                    'base_id': int(base_id),
                    'target_id': target_id,
                    'target_name': target_name,
                    'raw_predicted_name': names[int(raw_ranking[0])],
                    'raw_correct': int(raw_ranking[0] == target_id),
                    'raw_rank': int(np.where(raw_ranking == target_id)[0][0] + 1),
                    'loo_predicted_name': names[int(loo_ranking[0])],
                    'loo_correct': int(loo_ranking[0] == target_id),
                    'loo_rank': int(np.where(loo_ranking == target_id)[0][0] + 1),
                    'base_mean_predicted_name': names[int(mean_ranking[0])],
                    'base_mean_correct': int(mean_ranking[0] == target_id),
                    'base_mean_rank': int(
                        np.where(mean_ranking == target_id)[0][0] + 1
                    ),
                    'self_score_lift': self_lift,
                    'raw_target_margin': float(
                        matrix[target_id, target_id]
                        - np.delete(matrix[target_id], target_id).max()
                    ),
                    'loo_target_margin': float(
                        loo[target_id, target_id]
                        - np.delete(loo[target_id], target_id).max()
                    ),
                })

per_example = pd.DataFrame(per_example_rows)
self_lifts = pd.DataFrame(self_lift_rows)


def grouped_bootstrap(frame, value_column, iterations, seed):
    group_ids = np.array(sorted(frame.base_id.unique()))
    per_group = frame.groupby('base_id')[value_column].mean().to_dict()
    observed = float(frame[value_column].mean())
    rng = np.random.default_rng(seed)
    boot = np.empty(iterations, dtype=float)
    for index in range(iterations):
        sampled = rng.choice(group_ids, size=len(group_ids), replace=True)
        boot[index] = np.mean([per_group[int(group_id)] for group_id in sampled])
    return observed, float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))


def permutation_metrics(prompt_id, score_kind, iterations, seed):
    matrices = {
        int(base_id): matrix_for(scores, prompt_id, base_id, score_kind)
        for base_id in sorted(scores.base_id.unique())
    }
    observed = {'raw_top1': [], 'loo_top1': [], 'self_score_lift': []}
    for matrix in matrices.values():
        loo = loo_center(matrix)
        for row in range(len(names)):
            observed['raw_top1'].append(int(np.argmax(matrix[row]) == row))
            observed['loo_top1'].append(int(np.argmax(loo[row]) == row))
            observed['self_score_lift'].append(
                matrix[row, row] - np.delete(matrix[:, row], row).mean()
            )
    observed = {key: float(np.mean(value)) for key, value in observed.items()}
    rng = np.random.default_rng(seed)
    nulls = {
        key: np.empty(iterations, dtype=float) for key in observed
    }
    for iteration in range(iterations):
        accum = {key: [] for key in observed}
        for matrix in matrices.values():
            loo = loo_center(matrix)
            assignment = rng.permutation(len(names))
            for row, assigned_name in enumerate(assignment):
                accum['raw_top1'].append(
                    int(np.argmax(matrix[row]) == assigned_name)
                )
                accum['loo_top1'].append(
                    int(np.argmax(loo[row]) == assigned_name)
                )
                accum['self_score_lift'].append(
                    matrix[row, assigned_name]
                    - np.delete(matrix[:, assigned_name], row).mean()
                )
        for key in observed:
            nulls[key][iteration] = np.mean(accum[key])
    return pd.DataFrame([
        {
            'prompt_id': prompt_id,
            'score_kind': score_kind,
            'metric': key,
            'observed': observed[key],
            'permutation_p': float(
                (1 + np.sum(nulls[key] >= observed[key]))
                / (iterations + 1)
            ),
            'null_mean': float(nulls[key].mean()),
        }
        for key in observed
    ])


permutation_tests = pd.concat([
    permutation_metrics(
        prompt_id, score_kind,
        spec['permutation_iterations'],
        spec['seed'] + 1000 * prompt_index + 100 * score_index,
    )
    for prompt_index, prompt_id in enumerate(spec_prompt_ids)
    for score_index, score_kind in enumerate(('sum_logprob', 'mean_logprob'))
], ignore_index=True)

summary_rows = []
for (prompt_id, score_kind), group in per_example.groupby(
    ['prompt_id', 'score_kind']
):
    for metric, column, metric_chance in (
        ('raw_top1_accuracy', 'raw_correct', chance),
        ('loo_centered_top1_accuracy', 'loo_correct', chance),
        ('base_mean_centered_top1_accuracy', 'base_mean_correct', chance),
        ('raw_mrr', 'raw_rank', np.nan),
        ('loo_centered_mrr', 'loo_rank', np.nan),
        ('self_score_lift', 'self_score_lift', 0.0),
    ):
        work = group.copy()
        if metric.endswith('_mrr'):
            work['value'] = 1.0 / work[column]
        else:
            work['value'] = work[column]
        value, ci_low, ci_high = grouped_bootstrap(
            work, 'value', spec['bootstrap_iterations'],
            spec['seed'] + len(summary_rows) + 2000,
        )
        p_value = np.nan
        permutation_metric = {
            'raw_top1_accuracy': 'raw_top1',
            'loo_centered_top1_accuracy': 'loo_top1',
            'self_score_lift': 'self_score_lift',
        }.get(metric)
        if permutation_metric:
            p_value = float(permutation_tests.loc[
                (permutation_tests.prompt_id == prompt_id)
                & (permutation_tests.score_kind == score_kind)
                & (permutation_tests.metric == permutation_metric),
                'permutation_p',
            ].iloc[0])
        summary_rows.append({
            'prompt_id': prompt_id,
            'score_kind': score_kind,
            'metric': metric,
            'value': value,
            'ci_low': ci_low,
            'ci_high': ci_high,
            'chance_or_null': metric_chance,
            'p_value': p_value,
        })

# Greedy open-vocabulary exact generation for the first prompt only.
value, ci_low, ci_high = grouped_bootstrap(
    generations, 'exact', spec['bootstrap_iterations'], spec['seed'] + 9000
)
summary_rows.append({
    'prompt_id': generations.prompt_id.iloc[0],
    'score_kind': 'greedy_open_vocabulary',
    'metric': 'exact_accuracy',
    'value': value,
    'ci_low': ci_low,
    'ci_high': ci_high,
    'chance_or_null': np.nan,
    'p_value': np.nan,
})
summary = pd.DataFrame(summary_rows)

per_name = (
    self_lifts.groupby(['prompt_id', 'score_kind', 'target_name'], as_index=False)
    .agg(
        mean_self_score_lift=('self_score_lift', 'mean'),
        positive_bases=('self_score_lift', lambda x: int((x > 0).sum())),
        n_bases=('self_score_lift', 'size'),
    )
)
per_base = (
    self_lifts.groupby(['prompt_id', 'score_kind', 'base_id'], as_index=False)
    .self_score_lift.mean()
)
prompt_consistency = (
    per_base.pivot_table(
        index=['score_kind', 'base_id'], columns='prompt_id',
        values='self_score_lift'
    ).reset_index()
)
if len(spec_prompt_ids) == 2:
    prompt_consistency['both_positive'] = (
        (prompt_consistency[spec_prompt_ids[0]] > 0)
        & (prompt_consistency[spec_prompt_ids[1]] > 0)
    ).astype(int)

print('Candidate-free receiver summary')
display(summary.sort_values(['score_kind', 'prompt_id', 'metric']))
print('Within-function permutation tests')
display(permutation_tests)
print('Per-name self-score lift')
display(per_name)
print('Cross-prompt per-base consistency')
display(prompt_consistency)
print('Open-vocabulary generation examples')
display(generations.head(20))

# Plot 1: accuracy with and without pre-specified prior centering.
plot_rows = summary[
    summary.metric.isin([
        'raw_top1_accuracy',
        'loo_centered_top1_accuracy',
        'base_mean_centered_top1_accuracy',
    ]) & (summary.score_kind == 'mean_logprob')
].copy()
labels = [
    f"{row.prompt_id}\n{row.metric.replace('_accuracy', '')}"
    for row in plot_rows.itertuples(index=False)
]
fig, axis = plt.subplots(figsize=(11, 5))
axis.bar(range(len(plot_rows)), plot_rows.value)
axis.axhline(chance, linestyle='--', label='chance')
axis.set_xticks(range(len(plot_rows)), labels, rotation=25, ha='right')
axis.set_ylabel('Top-1 accuracy')
axis.set_title('Candidate-free identifier scoring')
axis.legend()
axis.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULT_DIR / 'candidate_free_accuracy.png', dpi=160)
plt.show()

# Plot 2: self-score lift by prompt.
lift_plot = summary[
    (summary.metric == 'self_score_lift')
    & (summary.score_kind == 'mean_logprob')
].copy()
fig, axis = plt.subplots(figsize=(8, 5))
positions = np.arange(len(lift_plot))
errors = np.vstack([
    lift_plot.value - lift_plot.ci_low,
    lift_plot.ci_high - lift_plot.value,
])
axis.bar(positions, lift_plot.value, yerr=errors, capsize=5)
axis.axhline(0, linestyle='--')
axis.set_xticks(positions, lift_plot.prompt_id)
axis.set_ylabel('Mean token log-prob self-score lift')
axis.set_title('Receiver response without visible candidates')
axis.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULT_DIR / 'candidate_free_self_score_lift.png', dpi=160)
plt.show()

# Plot 3: mean LOO-centered score matrix for each prompt.
for prompt_id in spec_prompt_ids:
    matrices = [
        loo_center(matrix_for(scores, prompt_id, base_id, 'mean_logprob'))
        for base_id in sorted(scores.base_id.unique())
    ]
    mean_matrix = np.mean(matrices, axis=0)
    fig, axis = plt.subplots(figsize=(8, 7))
    image = axis.imshow(mean_matrix)
    axis.set_xticks(range(len(names)), names, rotation=45, ha='right')
    axis.set_yticks(range(len(names)), names)
    axis.set_xlabel('Scored identifier')
    axis.set_ylabel('Encoded identifier')
    axis.set_title(f'LOO-centered score matrix — {prompt_id}')
    fig.colorbar(image, ax=axis)
    plt.tight_layout()
    plt.savefig(
        RESULT_DIR / f'loo_centered_matrix_{prompt_id}.png', dpi=160
    )
    plt.show()

scores.to_csv(RESULT_DIR / 'candidate_free_scores.csv', index=False)
baseline_scores.to_csv(
    RESULT_DIR / 'candidate_free_base_mean_scores.csv', index=False
)
generations.to_csv(
    RESULT_DIR / 'candidate_free_greedy_generations.csv', index=False
)
per_example.to_csv(
    RESULT_DIR / 'candidate_free_per_example.csv', index=False
)
self_lifts.to_csv(
    RESULT_DIR / 'candidate_free_self_score_lift.csv', index=False
)
per_name.to_csv(RESULT_DIR / 'candidate_free_per_name.csv', index=False)
per_base.to_csv(RESULT_DIR / 'candidate_free_per_base.csv', index=False)
prompt_consistency.to_csv(
    RESULT_DIR / 'candidate_free_prompt_consistency.csv', index=False
)
permutation_tests.to_csv(
    RESULT_DIR / 'candidate_free_permutation_tests.csv', index=False
)
summary.to_csv(RESULT_DIR / 'candidate_free_summary.csv', index=False)

preflight = json.loads(
    (ARTIFACT_DIR / 'preflight.json').read_text(encoding='utf-8')
)
metadata = {
    'notebook_version': spec['notebook_version'],
    'created_utc': pd.Timestamp.utcnow().isoformat(),
    'spec_without_samples': {
        key: value for key, value in spec.items() if key != 'samples'
    },
    'embedded_sample_count': len(spec['samples']),
    'embedded_sample_repos': [sample['repo'] for sample in spec['samples']],
    'preflight': preflight,
    'package_versions': {
        package: (
            importlib.metadata.version(package)
            if any(
                dist.metadata.get('Name', '').lower() == package.lower()
                for dist in importlib.metadata.distributions()
            )
            else 'unknown'
        )
        for package in (
            'transformers', 'accelerate', 'huggingface_hub',
            'pandas', 'scipy', 'torch',
        )
    },
    'primary_question': (
        'Does Model B assign higher likelihood to the identifier actually '
        'encoded in a latent packet when no candidate list is visible?'
    ),
    'pre_registered_primary_metric': (
        'Within-base, within-candidate self-score lift under each of two '
        'candidate-free prompt paraphrases.'
    ),
    'interpretation_boundary': [
        'The receiver prompt never displays the eight nonce identifiers.',
        'All selected identifiers have identical token counts under both pinned tokenizers.',
        'Teacher-forced scoring still evaluates a researcher-defined set of eight names; it is not open-vocabulary generation.',
        'Positive self-score lift supports weak receiver use because lexical prior is removed by comparing the same candidate across same-function interventions.',
        'LOO-centered top-1 is pre-specified but remains a diagnostic readout, not deployment accuracy.',
        'A null result would show that this candidate-free test does not support functional receiver tracking, while leaving external linear accessibility unchanged.',
        'The twelve samples and one adapter/receiver pair do not establish a universal property of continuous latent channels.',
    ],
}
(RESULT_DIR / 'metadata.json').write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False, default=str),
    encoding='utf-8',
)
(RESULT_DIR / 'samples.json').write_text(
    json.dumps(spec['samples'], indent=2, ensure_ascii=False),
    encoding='utf-8',
)

for log_path in LOG_DIR.glob('*.log'):
    shutil.copy2(log_path, RESULT_DIR / log_path.name)

(RESULT_DIR / 'README.md').write_text(
    '# Qxern candidate-free identifier receiver probe\n\n'
    'Start with `candidate_free_summary.csv`, then inspect '
    '`candidate_free_permutation_tests.csv`, '
    '`candidate_free_prompt_consistency.csv`, and '
    '`candidate_free_per_example.csv`.\n\n'
    'The primary test is self-score lift: for the same base function and the '
    'same candidate identifier, does Model B assign a higher likelihood when '
    'that identifier is the one encoded in the packet than when another '
    'identifier is encoded? The prompt contains no candidate list, and the '
    'selected names have matched token counts under both tokenizers.\n\n'
    'A positive effect across both prompt paraphrases would strengthen the '
    'claim of weak receiver use while leaving reliable open-vocabulary '
    'recovery unresolved. A null effect would weaken the prior closed-set '
    'receiver-use interpretation.\n',
    encoding='utf-8',
)

zip_path = Path(shutil.make_archive(
    str(WORK_DIR / 'qxern_v6_candidate_free_identifier_probe_results'),
    'zip',
    root_dir=RESULT_DIR,
))


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


print('Results ZIP:', zip_path)
print('SHA-256:', sha256_file(zip_path))

try:
    from google.colab import files
    PUBLIC_RESULT_ZIPS.append(str(zip_path))
except Exception:
    print('Download manually from:', zip_path)


Candidate-free receiver summary


                prompt_id              score_kind  \
24       exact_identifier  greedy_open_vocabulary   
2        exact_identifier            mean_logprob   
4        exact_identifier            mean_logprob   
1        exact_identifier            mean_logprob   
3        exact_identifier            mean_logprob   
0        exact_identifier            mean_logprob   
5        exact_identifier            mean_logprob   
14  recover_function_name            mean_logprob   
16  recover_function_name            mean_logprob   
13  recover_function_name            mean_logprob   
15  recover_function_name            mean_logprob   
12  recover_function_name            mean_logprob   
17  recover_function_name            mean_logprob   
8        exact_identifier             sum_logprob   
10       exact_identifier             sum_logprob   
7        exact_identifier             sum_logprob   
9        exact_identifier             sum_logprob   
6        exact_identifier             sum_logp

Within-function permutation tests


                prompt_id    score_kind           metric  observed  \
0        exact_identifier   sum_logprob         raw_top1  0.135417   
1        exact_identifier   sum_logprob         loo_top1  0.281250   
2        exact_identifier   sum_logprob  self_score_lift  0.118891   
3        exact_identifier  mean_logprob         raw_top1  0.135417   
4        exact_identifier  mean_logprob         loo_top1  0.281250   
5        exact_identifier  mean_logprob  self_score_lift  0.023778   
6   recover_function_name   sum_logprob         raw_top1  0.135417   
7   recover_function_name   sum_logprob         loo_top1  0.416667   
8   recover_function_name   sum_logprob  self_score_lift  0.259412   
9   recover_function_name  mean_logprob         raw_top1  0.135417   
10  recover_function_name  mean_logprob         loo_top1  0.416667   
11  recover_function_name  mean_logprob  self_score_lift  0.051882   

    permutation_p  null_mean  
0        0.316737   0.125117  
1        0.000400   0.12483

Per-name self-score lift


...

Cross-prompt per-base consistency


prompt_id    score_kind  base_id  exact_identifier  recover_function_name  \
0          mean_logprob        0          0.006131               0.037389   
1          mean_logprob        1          0.031619               0.051820   
2          mean_logprob        2          0.012004               0.038450   
3          mean_logprob        3          0.007582               0.041084   
4          mean_logprob        4          0.033055               0.083583   
5          mean_logprob        5          0.026490               0.031455   
6          mean_logprob        6          0.053972               0.058573   
7          mean_logprob        7          0.027268               0.086521   
8          mean_logprob        8          0.032317               0.048724   
9          mean_logprob        9          0.043950               0.066139   
10         mean_logprob       10          0.025812               0.049953   
11         mean_logprob       11         -0.014858               0.028898   

Open-vocabulary generation examples


           prompt_id  base_id  target_id   target_name  \
0   exact_identifier        0          0  qzx_homipamu   
1   exact_identifier        0          1  qzx_lozanoli   
2   exact_identifier        0          2  qzx_jadayaca   
3   exact_identifier        0          3  qzx_hapisaco   
4   exact_identifier        0          4  qzx_kelowajo   
5   exact_identifier        0          5  qzx_laneveji   
6   exact_identifier        0          6  qzx_muxocuka   
7   exact_identifier        0          7  qzx_jupalore   
8   exact_identifier        1          0  qzx_homipamu   
9   exact_identifier        1          1  qzx_lozanoli   
10  exact_identifier        1          2  qzx_jadayaca   
11  exact_identifier        1          3  qzx_hapisaco   
12  exact_identifier        1          4  qzx_kelowajo   
13  exact_identifier        1          5  qzx_laneveji   
14  exact_identifier        1          6  qzx_muxocuka   
15  exact_identifier        1          7  qzx_jupalore   
16  exact_iden

Results ZIP: /content/qxern_candidate_free_identifier_probe/qxern_v6_candidate_free_identifier_probe_results.zip
SHA-256: 9bdba42f6feb1159eb9bdbb5aed9f3640ed660bf6b9b05e1e73c7766760ce202


---
# Part III — Pairwise fine-grained behavior sensitivity

This section uses natural code-edit pairs with identical names, signatures, and sidecars. It tests whether packet swaps move the relative likelihood of two behavior descriptions in the expected direction. Multi-turn stress runs only if enough pairs pass the fresh-state gate.

In [14]:
# 2. Write the experiment specification and isolated worker.
import base64, gzip, json
from pathlib import Path
WORK_DIR = Path('/content/qxern_pairwise_state_transition_gate_v1')
ARTIFACT_DIR = WORK_DIR / 'artifacts'
LOG_DIR = WORK_DIR / 'logs'
RESULT_DIR = WORK_DIR / 'results'
for directory in (WORK_DIR, ARTIFACT_DIR, LOG_DIR, RESULT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
SPEC_PATH = WORK_DIR / 'spec.json'
WORKER_PATH = WORK_DIR / 'worker.py'
SPEC_PATH.write_bytes(gzip.decompress(base64.b64decode('H4sIAMRScWoC/7VYbW/bNhD+K4SBYXZgu/KbrARNga7bt63Y1n5LA4GSThEXSdRIKnZW5L/vjpRsR5LTdi8Bmtjk3XPH5453x34eldJAJOV9+ABKC1mOrthoMfdGUzbSAAl+XS/xcyETyEMeCloZ/baD8hX9Ws43s3e4pWaL+eaH0YmgggfR4iXpKgZ/G3uwXK8DL116az+IfEiXweXaXy4T/xKi7So66kddQys05M2DH05FTk0s09jzV/56u9hEl/42XQS+x4Ffxukq2G6DYLvZBEG02JJ+VJdJDqheSVLle1GIV3/uQZWzB3+WQJXLx5kozfqZ8NEWbD2Pp8sk3Ub+Itgs+GqxXqZ0BPC2cQQBeItV6ltbPOGVARWmIoeSF2ANujXtbIatyGpp5P28MqdqOuPLjU9KUbDZbJYLDwIvWXp+vOJb8JHI1SrxLnniw3LjIZlbH7/FyXobXMbrYJmu8eTe5TYOLHF8H8ZIXoiGoNQIi+B0RimNNopXoUCj3OA5aXPh4Q9uV6CK2tjl5wIbt1+IMlQyqrUJKy6UDlOJnhsFmoRWjeGBLbJdKVlUJjRQVDk3QMs3n0cu+pWCWGgII8j4g5CKzmBgb2jvLfv10WSyZGldxuQQE5phSBEaSgMJix6ZyYARaGnYA8RGKs0iyOXu6lP5+vXrn99+/On9xw9v3rz5VP4IOlYiAiaMZo1Z1pplomSyBKYzqQyz8GUM89HTlLWeCvQeCtxxLEGaor1Td38Hnlh/YjwuHR89pFAcXbaa5/z7gNvAYM9jkz+yXcaNBTucXYGpVakZOhtnvLwDfXD66O4t0U1RcBzTx+aamQx9yGSehJHEfOfqkXy3qcLtBYYUgXI8U6hjibJj92fKDprX3gTdZvjjfGE3VoRhwJn7hA45LSbSZunNUf/2U3mwGf2vNq97RhWkoIgkd9rfGzIbTUxXYWm/U4BRUKjOS0v/AWf+HCUaQHmmjP7BnzXPmZEDQJAImzOwj/NaiwdgVJtrCmm7cFBwSXgSSrxH5jFE8nidm4Eo8hS/IR6/g/EDz2vQHQ51XbQb7BXDqnX4hhS6jwxyvB7e3BuM2X9n4T0m8IsRIu64EiYrwIiYFcDLKUOGyjv2FyhpEwHptpwge1VtXorUC2glXaUzaIdw0frMrrOU53nE4/teeDSWkDARWGHo3vbjIxU2U+w2mDSJHjd/u/whBiTt5pTdw+N1zoso4cytXTV/b763yff97WQoUP+PqSkuULrC9UdVw2QwfB8Q1dHdAFOxPlxYrmMoE2LdejgUsZcBEhhAaKN0RG8uVVe6F7IC1B2E1BQgIRf6QXMSGoxBFD2OuAZMnCqhftYh9PPFhdu+uGgEnoZi822IzQaBkuTTIOm/EKQjreBVRbgY3ga0vXYSSVEC2xLhNItDATgDdqJ1hHpm4VksGlV2wm3vvuQCbWJ8zjaknSgTuRvjYFIgAznOcuZ60+HIbt5c2c2XGs3XY83s5tUtlSz70VWsm5c7Sl21FT/FNmwaTYv4UmE6quV8QKulE2lMxb7Na12n9M2djGpNl9oYgxXi1FdqLGxFn98SV3ku/oIw5ziWjGmU6XBBS3Pqj9V4MsfBBdR4sNR8K1RdVUeobvEgGTqeUjSrUP7sMiRDVzzG+l3SXFUiB02BIFxiz3pHRx6sJ/8M07p5wGzDcLDURuIg1gsBphvgZCwxyQcmrjiTEkNEgW5T0iq0zAkitkmEZumEUNc8rRwpsWu8qfsxzus4ro/t0tT2Xqs+YTO2mEwGEt1KDt6Z/8q9vjn23Ylnt1+cAEiO4USMvdmpx9iiKhywm0uDlUckTNFY/KX234Pa4auIoLhNDCujcXRzE/VJ0FuTTchbNQuC6dSLfAJJXcHZ0NelQCOWW93w8Py65EKbcYJz6TzFBxS25VZs8P79E7imjSct7uTm6mq2OBeNAss99tK6wvNQL6L7kwOzbxv1QBfqWPVkHONFO1DYC8fXYdlS2IFqo+HsNLHoCPZCUdbYbUVMT3Qc0AeuIUbehGnOsRHb3wPjLL6gc7s5sVMifaJJxIkPXp6vAHUomtEgdRa2E4l3hKsZjp/MqNpkj207Fu7JcnYIbhRlmR9UDD0zuTq+OxHxOc+tCUe0FWNEBU7O/YwX2h1a1fnADNVuQ4KjFvb8sf09Pax3+LG1bI7VzIph9TrIDbH978Av2Hgx905tnOlMdYTtNG6aRCuM8yzjmPl7qiKFXaE7ZmUs/lA03laY/DAIRBboyS/TDsQx+ffHQlQpWeGwjAr42myRerFpX/2hLWP94Nj90KaFe6tNqZ9OWSbusgHu7B61GdqfunQaLkzfAGy37bgld+z1dfMdP5DsF96L7+hYum0GqNZ0BoEpoXDFRYSA8WEta8Qj0C81Cwdlb8wuA+pgzAZth0/I5rrZ82FjlPhF7eg/legl2e8cbbgU/IHT2kzW+C+d2ZbFmh7RjZmb9sKCmzgbeJLQMujQ8Ds7bU0ZfhoeuTiONeTymCQGHyNfi4VjehepG4gM4nsq6ECEHIcp54Vlzi3yu8EidUadDPeVX5yLHW9Pt09/A3VmSL8DFwAA')))
WORKER_PATH.write_bytes(gzip.decompress(base64.b64decode('H4sIAMRScWoC/9U8/Y/bNrK/71+hy+FBUuJVd5PeoTDq4KVNcvdwaZpr0gccHEOgLXqtriyporQf3e7//maGQ4qUZe/HpQe8BRJLFDkczvcMKa2bahuk6bpru0amaZBv66ppA1GWVSvavCrV0ZFpa85q0Shp71VrLjdCbYp8aW5/UVVpritlrho7ss238miNM9eixYFm2g9wa+erVdfmhR1TNauNd5OUJeAQlHaqthGlWlfNVjZKg990Z2d5ebYWK5luOjvNZo13aVZdlkUlMt3XHW06vura6vuqXOdnE7r+ocpk8bZqvhedEsW7H3Trp+pclvlvsjk6OsrkOlClqNWmaqNCLGUxDVTbxMHxy+B9VcrpUQB/F9tgxutLLvKm7USRbuW2aq6jmDrUTbXqu3yAO6lUVKnkTLZ1nkUxdxPXuADoeUP3+BfSrOE0oN9J366uVSu3qbgQOTwqZHq2hF4X28S2BF8FpyfPv3769IUzrNaTpw38oxHYkGhs07xcV1GcwLPRsasuE6koimolWpnp4Zp3+MQAsR2iuIcS5Gu3a656xKGbLJQkeg4na6SSzcX+uczzR091y/zJyzZCOU+ybluriDkRT4J10anN7FPTydjIw0Y8/8tf03UO8FDgSSKC30naSTDgVstFlp9J1QI7WaESPZSF4jJvN0FVy5KgTIKwWYYxqsBGlFnBooV/IMXBatOV50FeBnkrG5DE7TITU+6ZNFJkEa4+eEpEALSXYRj3EHpkkq7OgDcRwdN4NBKMRWmeb+SVvopwvatCgCy8gxFl+8+3pE1RWSagOF0heQKkSQqik7dpGilZrCcAawsN+rfq2klQdtu0Rb1S0ChrXC4+2+RZJqHfel3aa7iC5ajZN9ClqWoYPjtJTp3FqK4GLOLEzhnbR8D5HmzwXwaWTwhuBLaU8qqNNkTfDdI2Ov3rJIB5v54EzyfBabwDbxPMZsFJPx+uNvm1k00uCV6ZfBCN2ErkkRZBsEJZGXnLtwBjYNdJcvJ8AA8U8hdYmob3Li+laCKXoDzYH7QEnTtnHDR33uXEQ1eIUlwkYHQmI2LCQEIcQImoQTCzntev81Ub3XjdtZY2lVKgmtixK9ocSfuqRXEBZxO5PGay91zlX5BV0a42oE+NarWiTXbnKUH2TvU878S1bN7DvQN+bAgu5z+E2fOHYQbirgd8lCA8gJAoIp/Xu4oBOg1d/vbm3c+RvnytkYwYWd3IIFx9clDZh/6Le6N/G48JK0y/I63u9IiePwxndYZ5E1Nva1lAbi9Fk7Fh2aQr8NmTQLRtmW6FOndEWLsEgOhqUaRHJGsw523kYI9aazuzCiddqeBS/iajkziRVzUorwGgcmqdBMen+K8HVIssg6CEsAF4FjMyFZ76kWqhCjqK5mugQPnMZDYBVZ3p/nNWsUVEGE94mf3vubxOXRxm7k3sgTdrZsBapxhw8MzOHt+NE+mWRcn9uXPG53fOODLohTOIm1GJuNFhKzszT8giT1LtEO3N0c2nIhM12mxVy9UkIEuNfuoiXxknhy4aHbkfakaNrKs0z2Y4cB4uO3THKTaGC1BeiBBK8Ab8lCdJTTN2aWAOBfZoCEC3Yg9CZkb/61WKFcWXEEqg9A5iEd0F3JbT60/QzUNADwoXvew1Ioeo6KeuxFD+TdNUYEKevNLdIfSQIKjdNtjmaosGcRrc9OBvEf7N+AS3TzQ+bXPtuG9IRCSgrv0jUVHHP1tRpxQ5Ij3CVd2Fk+BS5mebVqVVWVxzEIYw5NUK/Ffw6brW2D4KuqUV0LqEceVKRjQcjdaqhVCszHaeYcQehXSZYq8w5t67KNDv3O27oD6rQopSUpSvo0+0DaDFk+BCFJ0kA0FTQai3VZEDmUbCOOjsGZa6kev8iiKYcEvuOgHaGXbQ9a86FUqGUSEsn6AmMGXTKoxJIw1u0NGdn37nBUau3HXh4yizOf0uoDMt6sglDHfh4PPXLm+IGlHIZhjxZfOdaAGwLaC/ThPHKicJ2ILkhXmg+QrCCqw7A7hzIJehMQVAZkpYOzZBamxpvjAywcOHKvIPec3qEXrqUVeQQNg5AaoCJeHbW8ZoPAjsBcWsfoFpQi05g4AMD7pgesLdBpTh3vPTRWxHLEdGOJQzQ054CJERrBiGj3Mct03Omqqro9M4dgmn+QfEibbBdAZ0TMgcRA0z4nMSfc6exZ9R4GBIHC/Yi9dkO7fiKrJTxWDIT7X021jFx9pj7sk+3FnGYaifpBDhNJ2XD0g/4qStIm34oUsL9mWmLQnFDy84VCefAaknhJXo8VnNqLXXdmNLIBuEO8d4mf4SNIMDdFJikzkgqU0f2+jZAduasCxDKt/A5NFbAdmtl9UxIL3eniC9/Z64q2GfCDdVcSHTFvIjiH6wYBLpH0YCjKAmOlyAD2/4KagkNLjDQBIw17amdiUKnYhrCLGf5J5DpHemyI7dhJnEuKuB0BRJdzsBazkwSZ5bMX/ahs4YxejpUw003ukI2LC9VWQC+nLO8I9J2Rsy87ffDfWcUsplxw69RmgVYLpPLcwOLZopRaAeG5zBwewQ144GlCEcnH6AiDPJkGmWTD6JMtGiadIQQG+01PeENvNhP+023TlomVH/zCUEPQN2j4y6AydtpSHNa/Jl13IoE35P9U9DIKZn4ABk7qD1IcAmOKybfCuaa5j7rBRYUo1QJnUJkFnQSFyjUG1C1VTqwErdlSsquaJZLaGZZJwuQMJxYLKssutBBFJSfhMhwLcM4LWEzAcbXqnrcuW0GgsLENAZ2wljjMlOh1T5X1wZU+TNFQRsLfhAeQWmoLgOgIoQM9XHhbyQhUWdybNG02yhg+3Vdgx1FdrLBK+OrGHCxsjOPRf4mJYucN1G+nAMevVKYWyHdyB280XcRxLPdsZiL5rM7RTNw6ch5hL48AIQaM6wz4Ki4L5NF/7mi/gu+OeXBqHhLP00ZFP8WajJnyQ2SawA04pUQSZRZ9OGXhAb51eExJWDRGrHwQxXrpFaeDIL5MdkYsK0n9j5rC2HBAtC8hzsqWh25Hd8LNaqR0Xfn/oJTnCDIG7BYAM3k18gDoo0tPg2Dn7vXRUGRBQx8kN4ZmaDJ+YSmzV0BSwhXZwG11I9gdX8t/bEebmWsKiVTCHUlWBvEAm4hxvIg1fnslURPik4d8JK/qT3g9hPYTpw5UQE2t/rag+ahdnXTB4qk5ICL6yjpki5r6SdTIiFBDd2gTjeihrJVkKnOQGYajDPnAG9vOVl3REX7Aoi6jZh4oCNL1XVqFlYYyDMGf8sxFUBMmftBlrbBjRW5zvoQfWa9dNZv3wn4ulVA4JFRoAoCV5UYzThJ6mxyRjiKAbfKZmuBITDMwpDLK6ZH/7gn433eKLEgzc/Pl2Ygk2fVxAnTF3SJOwmgtPoQQZqCnxU/QgXcQKZXhQnG1Gs3erPTsH/JtQilEG8sQUZNtxB1hLpY5fNMXruqhW0L2NbbwfbBY6q8H4ErEivgwKx2UnvZ+S6wPjWr0AwsSmxN7V2ZJTNb/VtRHsl0xO21na/KrRgj2k5oRd/7oSEffI+GUS4d9ZItK6Q86Z0pd9oS3AzLgVEWnBCkPRFulJAYpUKyARGqiDm4Z4yCEm2auHxtoJYG2nvEJyxWN4fi+UhLJaPwyIfi9pSQY5Z53S457DbY2l7LO+ozryG6xLxcsoyr4Kb0Vlvv7qhSSGI/m60y9J0WZqSDSzAxonOTirI/T8vZfki/YuzbWoD1z3h2P/Q7itHHrvDcVhX2j260OyF5k2qS0+++cUHlI0Sl/BOuaUsxSK467+w6zwkPyFCJw5QLC4HRyzdEUAcPQ2W1nC0H/jvhFvrJx8Q6RsNj1aGIncLRg1diArslOAKCTJX1RC24QhP7K8Ci8w+lgGTaB6C8rOj9Dr2zUuXbnsQfy3PZCkbTPxx+HR3EU9cSlKQQfQfxB37ia+HLA8O2aW+mYg4wCAey4UNHjuoymOAAyoGKaoBqZ64ZX4rj8YJ3Vgg02AAFPTE8hSeEk+pjeBSC6/g9sDWP7rDDnfXwuocw6uzuvN3wTHb1iY4xSiM9idCV1/TC/gPjdfUOxCRpOZBmjpb7oP67dT3CSHZEGhlv0H3S75fOmCw2lJQNYZjDOiD7tYUNcxui63k9LXK1HE8IcWpkdMSu+cYiB8VGEX2wZ49QJAOx5hBfOedhtA6X4N/qCH/FGWW4/54ijZ/mRd5ew1jdTzjnBdwHHNy2eStTrpHDhBgZIJSNXsOGFGEQXFa166PvzGG7iFnD8CFQQAtv3iYgEC9IMEGnuxH7aGY/7hDt5hAgp1pgXID40RWSrfeO7hJ7x3SMOJMg+FxoR1S9FbtD6NJL7tFdZlCcIsHjVLA54yfM/4Q/9azmxCE9+R2pJKJBx1opzLf1oWESKLljRGV1SLUk8RuhRKLFFgZ1liT4aGYO5XbpaTsQ0GYrYu0rNgQxffxzsFoJiTKQvhymWftJsgqqbNdCmtsiZfmC02Ul0knLoAMU3A5ahAppKh/VxMbNUgwTuTOBvbCL0TqKi4P5ep3PzIKBe0+hPGgIElYGe9AHmGt/dcNwLh1fZiL9NCd4KzGYDEKhx1NjyyZTAd1/QxacX7jaChFBkqN5cyoMgeyZhZdSB5pKFv3ha80/Qx8lYD1RomIxsz080nw1PTjnYzpYscnhtyFCEBXsDhDQEzZ+PJOL3XYyd2yxUE9UeJC9mbctbFHezJIHJGx13L7E/8Q6ZQWiP4Kz+Z4q95JHo+OPv7r46c3P6Qffvrxhw+fgAhP/lV1AdjpQAQQOch1VxBbAqFUjsXDNgneVgWYhaDdSMzGm1Ax3ijUoEym2pc8OXr36tOb95/SD+9eff/m7z++e/3mJ5gh/Pbbb/WDjy9fvgz9ChLEqm3kyMZWKrQ6yhZCnWL8TmXepsHGaoPMF9cEE1wn2CDUKwOxF0FTSRBZlnIcijGL9tZs72RJpx/bTV6eo3N19kDurtH/YXh5+9Rv6Ad6TIfnFEIuma2ffPs7yCd54N9f3mwhTK8gFVrcfi7xBtI0DKfgnroBQ35/+bl8QrZqi8bJMiN4Blx0QFnh+FwaftIicA/O5SbGL4MjD0PaRLrPaAUKCYGanYuCrQLzIdFeAmZz60waEVAg2RQStcZ1I7vFu54BrEfg3HaQRSGlfeS75FWrbyFWclMVGR4P9ryGdjq0T5yswY7SgcumV42UNjlRPyZ2VndvzgFsLHtEMPUWO57e0rf4JJ5ot0oNdtM1tj4XvaCHKp5JGNFdQlfj4qBSqRylztJEZgk5hmgXQjy6AKRMZMBMeoC6LjYCZRKc9HsSQ+QPbUi8rwKdMrgjgB8drFes9T67aHkdICYcB1ixMaXKPZGJjooxCkLHZx6YeIUejNZ7V10DUh6Yk1y2/ItmJ7PimGfIgPGlotI4DJhriLr6u3BzWuw4CCe8kqfFOrpbgd1KJ6FonTGqztyivfBPvO3sfdP/+2qwGorz1NAKHmqXIPJiZOlWPvD59Ohxq4Whw9UeLLWe2tPbABPchbEGEWd+mnyclc+onDU19UQSS6wP2Z4Q0RC3R1Rg4njR9ATNhFVmhu4smWE/A9f7uXyDzhmDh24FC+mrEEHUFyYEHQYkB78CbARaPF00j6dg4UExeZBLkfmN9iZT89IARq/GpUwDL87AfXbbm2IIry9f3S6Ymls8WbuHmDWmNlWnUlGqS2TaCHVNn3+bvCBbzaPZdHqYTapb4/GqP4hJHhlAEvRsrlY1LCPOE8NZ2+1BLB4Ztctqg9NodxtWjI9hlt9zpvATUEpVXbOSgZ+PcmnuMlhKyPr1qwtZEvwMzgOpW8pLoESNL4GYAUsJAfCEXA/2sITVCCWa/ExTRm9xx96iWlWNLvCMxSb6BBxkSKZ0pAaRSV9T4hNWd5vvpLfJ5HLIh4CH6eewNlSU1xEGV5icQq5Oh45pDE7mDqFDV4ez8O9NX7u+DK7AXdbtdaDoqPrKlOf7XUve1/Yn0lm53W/k018Y6jjIxvsR5fwvQ9c6GysBkc9d733S75brLfmRchH0M7NkMrO1Mkiriijql+dum3K0pvGK/bJKUZVnhoEzr4akz4XrXoBBpQ6AvxdE3XsQtjYo9kzLvlSxj/2WLUOWeMfawWcT2Knujqc5oW9/4B5WttPhdDCH9ei8xoeHbaDhkt4342OuM9Y5c0DfJaZ7OB85qaEpS38MC+YDgJMenUivOl7YoIGtM3RLPUaCZA34yEjZ05j3ZKa/Y+0j6kw8IWr7eDkbAXgwaNNUKOJMNQowtVyDhie1bNa6XI7vTh2NbfLrPXSm2Myh3mSA5My/vffO/0F8ZSFqBTzRx4dGcA6OzZpies3t5ITPYp5h3r1UzmkCaMvhh48Q4G2qqnWLJgjJd+zz1dOFIRe1BFaXY9rGCXGOFGLp9rXvt7yOXL/gaaMZo9wqosmweokV+pSJj+txcDoZoP8scJ+NCJo9kkcnkQzVtP7aaWk1fVqCC+/Lkw76XGykwU47EgS3mogubrvZBtI4YmES94iqM8QBmjWnNHYJPIr43EZMVT5R7uuKz2zfWy8NQNQnjlQN6MXxKx1F52iF2Mm80IleVesz9X3RBQMXt7yC959LDGkgsqDrnzm8+ZYKUnrAV3wdj+bmFKlipSHCGf3NXtPrTzMwa4PTrzqxxJ/51HRceJkQAsbjxzW9y3koxjGbvAcCHPRTEHANTk3xkvZYMbKSnkpNny/uaRgNSpnjIxhJ31IZBO8wUj7+s53lVKkSGHwaG4a7mEsptmp2SiesbNQwGw8mPOnz5cr212eaI7u0+QlQQ53n9bB4RkbTpKx6hxFLzQW/zILC7byY5O5BgghyTdGpTZOeAlz8KIBQqzzn6pyxZVQ/QphYPiRJ3t2rdF6XauRK5hcH35d63P7hH3EU52H7h3oq/drz7OFHYOzrIw6YR56K2T0SQ0CNNmjg99uC/LJk/UJbkC+eP3ALcliPnmhCsGBixw59zRp3m3zJ5A3uLNXaA5Y7X0MaD/FJ84ANcwNl8HaZD/vOl9ic7Qlno4yBzO0+l38Acrhms4F9UBF39m0ftGvLwRCjpTfRFnckkD8xOryPaw6mhWNRVL9tMjzfpes3Zt+W7kZ3bk2dR+/S7G7iPm77V0sau0hCbeQ01WTsLNXCf5me60L3Pgllv/SAdZGrlryzDj/4LA2KD4YTfFsXnUpt0cmAD0de2xvsaBPQE/y+wfjrLqaah2+mjRZL5/SyBlLBLBLf7HHQxq0N/SqhPfHkfHgjHp3VJneP2BW6ObHbwnOX686iF7e+Xgz/qMhzKGBEatyrENQLz+EZiSsIkQ4z0vTj/NgJxPf2Mh94If3J9LddiF98UuAx5wsOTyazvLWD6GZhSnssC6ZozLfwlA9ap+Zsgnti4fBsEHOjCRsbCBkFcYeUY3UN/IMOLjNDj5u6VtnfH5746VMCvr/T7TiLMYJ3rJkuzuEOnqfj0Oqp+H4pcGzmfYXhUSdKHs8h10pOHVWgQNczl97T07vkTEfSMOgeicrX31jFuzfHRk5zUByRYgG/kFQsAgJ5RMQ3WAfvAeyeQST2DzRx7D0BN8Z345Pgq4Ax0SYioS4hpwD3HOrKjRnvtO090OJMnOJ8vEqamt71tCcfnKcu4NvRQ4w2VIPcFL+DdShWU7KQK3244v9p7GZXAH2Juji1ivyF0SecdAK3k3m5YGgF9I72Vdy/MmZh8dcJuGtqZBXo4b1d9/hwksMx80WJ3RjMvBy4WzJzFMei7J1WwBBt5h/rd7y5U5SS60fGZQ+OyWB9bbWqCmcp+Pfn4M2vXQ5SjBtzOkY6ptxkKWBdkIepZBeEPQwSclSFhwjvGWDF+2OcE4xsTuL4HhMuv8iEpzjhqTPhn4P3gvdC6QMBlCzrstUr/Bzad/pLZ3YrDswJCFrLW3KY1lHqepBkPcx0kytQYYyERzef+6WgnOgCyz2ICEu693I/iq23ymXX0hLxe0a043yNOxve9qOSW4GBegBZe76971KVKOHiN5ndvdjwI9GeNThLwi+/bMjuqgv8BA99X1GA36lgmf0Oq9kS1ms0YEEGcmV3scWZwM+YIcXA3+B7t1oIlLhWwXcH6VJXuUJPw1jck/+n/y7/d/SLc1RCb9/RNDdQpaNBZjnToy+V84ychDv6wxKaPanMvVKYu1OXx0apI7nH0b6UiMiv4zDm3DCbcG/vTCf2zXMoxdifRtwefDFVR0njkaj7HtMDo0oGuzes3I/JSCjohRz40DSMR4BbsAKRLWA3isrE5oO0yavmrMN64Ad6Yj6hSjcJHjQV/BxsJb7kOwlWmwpDnNm8f+NVfzGIIjO8pEg2nJhVh4sDQI+PMQah0536Q0EOQfcMuKya82Og6/gg/lACj6UfHK0i75jn3k+W3nFo4+fXr0ZqxbgEP+DEz5Pqrw/gs/juiBMXBSD6cdhAcTevSssTLo26gkzZttDvkmzPYRy++w8EM++LyyvwF2l1PqzV40auSmR5kTcmov372/TTj/94874/pEUItfyZPYftU/dM1dg71T3aXznjSP7NmdJiDD5Jkgd990WsIWzKC5K63Q9Yy2UP9lAVey9055GZSMkRmGPp1n2AkrWwKQZRypvy6CjHL8DiG4hpSstKU9TwNOWVaXU/+j/5XTEwnVoAAA==')))
print(SPEC_PATH)
print(WORKER_PATH)


Pairwise state-transition specification and worker: PASS


In [15]:
# 3. Preflight, encode the 24 natural-edit states, and score the candidate-free fresh-state gate.
import os, subprocess, sys

def run_worker(task):
    log_path = LOG_DIR / f'{task}.log'
    command = [sys.executable, str(WORKER_PATH), task, '--spec', str(SPEC_PATH), '--work-dir', str(WORK_DIR)]
    with open(log_path, 'w', encoding='utf-8') as log:
        proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ.copy())
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
        rc = proc.wait()
    if rc != 0:
        tail = log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-120:]
        raise RuntimeError(f'{task} failed with exit code {rc}.\n' + '\n'.join(tail))

for task in ('preflight', 'prepare', 'fresh'):
    print(f'\n===== {task.upper()} =====')
    run_worker(task)
print('Fresh-state scoring completed.')


Pairwise fresh-state scoring: PASS
Pairs: 12
Prompt paraphrases: 2
Fresh score rows: 192
Greedy generation rows: 24


In [16]:
# 4. Analyse the pairwise gate, conditionally run the multi-turn comparison, package, and download.
import hashlib
import json
import shutil
import subprocess
import sys
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def run_worker(task):
    log_path = LOG_DIR / f'{task}.log'
    command = [sys.executable, str(WORKER_PATH), task, '--spec', str(SPEC_PATH), '--work-dir', str(WORK_DIR)]
    with open(log_path, 'w', encoding='utf-8') as log:
        proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
        rc = proc.wait()
    if rc != 0:
        tail = log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-120:]
        raise RuntimeError(f'{task} failed with exit code {rc}.\n' + '\n'.join(tail))


def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]


def exact_sign_flip_p(values):
    values = np.asarray(values, dtype=float)
    observed = abs(values.mean())
    n = len(values)
    if n <= 20:
        means = []
        for signs in product((-1.0, 1.0), repeat=n):
            means.append(abs((values * np.asarray(signs)).mean()))
        return (sum(x >= observed - 1e-12 for x in means) + 1) / (len(means) + 1)
    rng = np.random.default_rng(42)
    null = []
    for _ in range(5000):
        null.append(abs((values * rng.choice((-1.0, 1.0), size=n)).mean()))
    return (sum(x >= observed - 1e-12 for x in null) + 1) / (len(null) + 1)


def bootstrap_mean(values, iterations=10000, seed=42):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    boot = values[rng.integers(0, len(values), size=(iterations, len(values)))].mean(axis=1)
    return float(values.mean()), float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))

spec = json.loads(SPEC_PATH.read_text(encoding='utf-8'))
fresh = pd.DataFrame(read_jsonl(ARTIFACT_DIR / 'fresh_scores.jsonl'))
generations = pd.DataFrame(read_jsonl(ARTIFACT_DIR / 'fresh_generations.jsonl'))

# Convert each 2x2 score matrix into per-state margins and assignment gaps.
records = []
for keys, group in fresh.groupby(['prompt_id', 'context_mode', 'pair_index', 'pair_id', 'edit']):
    prompt_id, context_mode, pair_index, pair_id, edit = keys
    pivot = group.pivot(index='encoded_state', columns='candidate_index', values='mean_logprob')
    if pivot.shape != (2, 2):
        raise RuntimeError(f'Incomplete matrix for {keys}: {pivot.shape}')
    s00, s01 = float(pivot.loc[0, 0]), float(pivot.loc[0, 1])
    s10, s11 = float(pivot.loc[1, 0]), float(pivot.loc[1, 1])
    margin_a = s00 - s01
    margin_b = s11 - s10
    assignment_gap = 0.5 * (margin_a + margin_b)
    records.append({
        'prompt_id': prompt_id, 'context_mode': context_mode, 'pair_index': int(pair_index),
        'pair_id': pair_id, 'edit': edit, 'margin_a': margin_a, 'margin_b': margin_b,
        'assignment_gap': assignment_gap, 'both_states_correct': margin_a > 0 and margin_b > 0,
    })
pairwise = pd.DataFrame(records)

primary = pairwise[pairwise.context_mode == 'latent_only'].copy()
robust = primary.groupby(['pair_index', 'pair_id', 'edit'], as_index=False).agg(
    prompts_passed=('both_states_correct', 'sum'),
    min_assignment_gap=('assignment_gap', 'min'),
    mean_assignment_gap=('assignment_gap', 'mean'),
    min_state_margin=('margin_a', 'min'),
)
# Correct min across both A and B margins.
mins = primary.groupby(['pair_index', 'pair_id'])[['margin_a', 'margin_b']].min().min(axis=1).rename('true_min_state_margin').reset_index()
robust = robust.merge(mins, on=['pair_index', 'pair_id'])
robust['robust_pair'] = (robust.prompts_passed == len(spec['prompt_templates'])) & (robust.true_min_state_margin > 0)
robust = robust.sort_values(['robust_pair', 'min_assignment_gap'], ascending=False)
selected = robust[robust.robust_pair].head(int(spec['max_pairs_for_stress'])).pair_index.astype(int).tolist()
run_stress = len(selected) >= int(spec['min_robust_pairs_for_stress'])
selection = {
    'selection_status': 'stress_enabled' if run_stress else 'fresh_gate_only',
    'selected_pair_indices': selected if run_stress else [],
    'robust_pair_count': int(robust.robust_pair.sum()),
    'minimum_required': int(spec['min_robust_pairs_for_stress']),
    'gate_definition': 'Both state-specific mean-logprob margins are positive under both candidate-free prompt paraphrases in latent-only mode.',
}
(ARTIFACT_DIR / 'selection.json').write_text(json.dumps(selection, indent=2), encoding='utf-8')

summary_rows = []
for (prompt_id, context_mode), group in pairwise.groupby(['prompt_id', 'context_mode']):
    mean, low, high = bootstrap_mean(group.assignment_gap.to_numpy(), int(spec['bootstrap_iterations']), spec['seed'] + len(summary_rows))
    summary_rows.append({
        'prompt_id': prompt_id, 'context_mode': context_mode, 'n_pairs': len(group),
        'mean_assignment_gap': mean, 'ci_low': low, 'ci_high': high,
        'both_state_accuracy': float(group.both_states_correct.mean()),
        'positive_gap_fraction': float((group.assignment_gap > 0).mean()),
        'sign_flip_p': exact_sign_flip_p(group.assignment_gap.to_numpy()),
    })
fresh_summary = pd.DataFrame(summary_rows)

print('Fresh pairwise summary')
display(fresh_summary)
print('Per-pair robust gate')
display(robust)
print(selection)

stress_summary = pd.DataFrame()
stress_per_pair = pd.DataFrame()
if run_stress:
    print('\n===== CONDITIONAL MULTI-TURN STRESS =====')
    run_worker('stress')
    stress = pd.DataFrame(read_jsonl(ARTIFACT_DIR / 'stress_scores.jsonl'))
    stress_records = []
    for keys, group in stress.groupby(['pair_index', 'pair_id', 'edit', 'protocol', 'target_state']):
        pair_index, pair_id, edit, protocol, target_state = keys
        scores = group.set_index('candidate_index').mean_logprob
        own = float(scores.loc[int(target_state)])
        other = float(scores.loc[1 - int(target_state)])
        stress_records.append({
            'pair_index': int(pair_index), 'pair_id': pair_id, 'edit': edit, 'protocol': protocol,
            'target_state': int(target_state), 'target_margin': own - other,
            'target_correct': own > other,
        })
    stress_per_pair = pd.DataFrame(stress_records)
    comparisons = [
        ('transition_history', 'single_b', 'history_transition_minus_single'),
        ('transition_sanitized', 'single_b', 'sanitized_transition_minus_single'),
        ('poison_recovery', 'single_a', 'poison_recovery_minus_single'),
        ('transition_history', 'transition_sanitized', 'assistant_replay_minus_sanitized'),
    ]
    rows = []
    pivot = stress_per_pair.pivot(index='pair_index', columns='protocol', values='target_margin')
    for a, b, label in comparisons:
        if a not in pivot or b not in pivot:
            continue
        diff = (pivot[a] - pivot[b]).dropna().to_numpy()
        mean, low, high = bootstrap_mean(diff, int(spec['bootstrap_iterations']), spec['seed'] + 100 + len(rows))
        rows.append({'comparison': label, 'protocol_a': a, 'protocol_b': b, 'n_pairs': len(diff), 'mean_margin_difference': mean, 'ci_low': low, 'ci_high': high, 'sign_flip_p': exact_sign_flip_p(diff)})
    stress_summary = pd.DataFrame(rows)
    print('Conditional multi-turn comparison')
    display(stress_summary)
else:
    stress = pd.DataFrame()
    print('Fewer than the predeclared number of robust pairs passed. Multi-turn stress was intentionally skipped.')

RESULT_DIR.mkdir(parents=True, exist_ok=True)
fresh.to_csv(RESULT_DIR / 'fresh_candidate_free_scores.csv', index=False)
generations.to_csv(RESULT_DIR / 'fresh_greedy_generations.csv', index=False)
pairwise.to_csv(RESULT_DIR / 'fresh_pairwise_margins.csv', index=False)
robust.to_csv(RESULT_DIR / 'fresh_gate_per_pair.csv', index=False)
fresh_summary.to_csv(RESULT_DIR / 'fresh_gate_summary.csv', index=False)
(RESULT_DIR / 'selection.json').write_text(json.dumps(selection, indent=2), encoding='utf-8')
if not stress.empty:
    stress.to_csv(RESULT_DIR / 'stress_candidate_scores.csv', index=False)
    stress_per_pair.to_csv(RESULT_DIR / 'stress_per_pair_margins.csv', index=False)
    stress_summary.to_csv(RESULT_DIR / 'stress_comparisons.csv', index=False)

# Plots.
plot = fresh_summary[fresh_summary.context_mode == 'latent_only'].copy()
plt.figure(figsize=(8, 5))
positions = np.arange(len(plot))
errors = np.vstack([plot.mean_assignment_gap - plot.ci_low, plot.ci_high - plot.mean_assignment_gap])
plt.bar(positions, plot.mean_assignment_gap, yerr=errors, capsize=5)
plt.axhline(0, linestyle='--')
plt.xticks(positions, plot.prompt_id, rotation=20, ha='right')
plt.ylabel('Mean pairwise assignment gap')
plt.title('Candidate-free fine-grained state discrimination')
plt.tight_layout()
plt.savefig(RESULT_DIR / 'fresh_assignment_gap.png', dpi=160)
plt.close()

heat = primary.pivot(index='pair_id', columns='prompt_id', values='assignment_gap')
plt.figure(figsize=(8, 7))
image = plt.imshow(heat.to_numpy(), aspect='auto')
plt.yticks(range(len(heat.index)), heat.index)
plt.xticks(range(len(heat.columns)), heat.columns, rotation=20, ha='right')
plt.colorbar(image, label='Assignment gap')
plt.title('Pairwise fresh-state margins by prompt')
plt.tight_layout()
plt.savefig(RESULT_DIR / 'fresh_pair_prompt_matrix.png', dpi=160)
plt.close()

if not stress_per_pair.empty:
    stress_plot = stress_per_pair.groupby('protocol', as_index=False).target_margin.mean().sort_values('target_margin', ascending=False)
    plt.figure(figsize=(10, 5))
    plt.bar(range(len(stress_plot)), stress_plot.target_margin)
    plt.axhline(0, linestyle='--')
    plt.xticks(range(len(stress_plot)), stress_plot.protocol, rotation=25, ha='right')
    plt.ylabel('Mean target-state margin')
    plt.title('Conditional single-turn versus multi-turn state readout')
    plt.tight_layout()
    plt.savefig(RESULT_DIR / 'stress_target_margins.png', dpi=160)
    plt.close()

preflight = json.loads((ARTIFACT_DIR / 'preflight.json').read_text(encoding='utf-8'))
metadata = {
    'notebook_version': spec['notebook_version'], 'spec': spec, 'preflight': preflight,
    'selection': selection,
    'primary_question': 'Can the native Qxern receiver distinguish two natural, same-signature code states through candidate-free pairwise likelihood scoring?',
    'secondary_question': 'For pairs that pass fresh scoring, does an equivalent multi-turn history reduce the current-state margin relative to single-turn input?',
    'interpretation_boundaries': [
        'The behavior descriptions are never shown in the receiver prompt; they are teacher-forced by the evaluator.',
        'The primary within-pair assignment gap cancels much of the fixed lexical prior of the two descriptions.',
        'A positive pairwise result supports receiver sensitivity to the edit but does not imply reliable free generation.',
        'The conditional stress is run only when at least three pairs pass both prompt paraphrases.',
        'This remains a one-way released-path history test, not true bidirectional latent recurrence.',
    ],
}
(RESULT_DIR / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
shutil.copy2(SPEC_PATH, RESULT_DIR / 'spec.json')
shutil.copy2(WORKER_PATH, RESULT_DIR / 'worker.py')
for log in LOG_DIR.glob('*.log'):
    shutil.copy2(log, RESULT_DIR / log.name)

readme = f"""# Qxern-v6 pairwise fresh-state and conditional transition probe

Selection status: **{selection['selection_status']}**  
Robust fresh pairs: **{selection['robust_pair_count']} / {len(spec['pairs'])}**  
Selected stress pairs: **{selection['selected_pair_indices']}**

Start with `fresh_gate_summary.csv` and `fresh_gate_per_pair.csv`.

The primary test uses two natural code variants with identical function names and signatures. The receiver prompt contains no candidate behavior descriptions. The evaluator scores the two descriptions under each packet and computes the diagonal assignment gap. This separates fine-grained packet sensitivity from free-form generation quality.

If at least three pairs pass both prompt paraphrases, the notebook automatically compares equivalent single-turn and multi-turn state readout, plus sanitized-history and wrong-answer-history conditions. Otherwise it returns a diagnostic ZIP without running an uninterpretable stress test.
"""
(RESULT_DIR / 'README.md').write_text(readme, encoding='utf-8')

checksums = []
for path in sorted(RESULT_DIR.iterdir()):
    if path.is_file() and path.name != 'SHA256SUMS.txt':
        checksums.append(f"{hashlib.sha256(path.read_bytes()).hexdigest()}  {path.name}")
(RESULT_DIR / 'SHA256SUMS.txt').write_text('\n'.join(checksums) + '\n', encoding='utf-8')

zip_path = Path(shutil.make_archive('/content/qxern_v6_pairwise_state_transition_gate_results', 'zip', root_dir=RESULT_DIR))
print('Result ZIP:', zip_path)
print('SHA-256:', hashlib.sha256(zip_path.read_bytes()).hexdigest())
try:
    from google.colab import files
    PUBLIC_RESULT_ZIPS.append(str(zip_path))
except Exception as exc:
    print('Automatic download unavailable:', exc)


Fresh pairwise summary


               prompt_id                   context_mode  n_pairs  \
0  implementation_effect                    latent_only       12   
1  implementation_effect  latent_plus_identical_sidecar       12   
2       precise_behavior                    latent_only       12   
3       precise_behavior  latent_plus_identical_sidecar       12   

   mean_assignment_gap    ci_low   ci_high  both_state_accuracy  \
0             0.005370  0.000522  0.010756                  0.0   
1             0.000617 -0.003805  0.005458                  0.0   
2             0.010575  0.000740  0.022284                  0.0   
3             0.006349 -0.002457  0.019009                  0.0   

   positive_gap_fraction  sign_flip_p  
0               0.416667     0.117891  
1               0.500000     0.713449  
2               0.500000     0.122285  
3               0.416667     0.439102  

Per-pair robust gate


    pair_index             pair_id                                     edit  \
9            9       discount_rule       fixed versus proportional discount   
8            8      numeric_filter              truthy versus exact boolean   
3            3    merge_precedence                       mapping precedence   
6            6        index_policy          clamped versus wrapped indexing   
7            7       dedupe_policy             first versus last occurrence   
5            5      case_transform               lowercase versus uppercase   
10          10      boundary_clamp  clamp versus reject-out-of-range policy   
0            0  threshold_boundary     exclusive versus inclusive threshold   
4            4           slice_end           prefix versus suffix selection   
11          11        prefix_match               prefix versus suffix match   
2            2      sort_direction        ascending versus descending order   
1            1       empty_default                  

{'selection_status': 'fresh_gate_only', 'selected_pair_indices': [], 'robust_pair_count': 0, 'minimum_required': 3, 'gate_definition': 'Both state-specific mean-logprob margins are positive under both candidate-free prompt paraphrases in latent-only mode.'}
Fewer than the predeclared number of robust pairs passed. Multi-turn stress was intentionally skipped.
Result ZIP: /content/qxern_v6_pairwise_state_transition_gate_results.zip
SHA-256: 960dd00194bca4bb08e93570c3d730adcc2062c08a556c1ae4fc8dcbea34723b


In [17]:
# Package the three public result archives into one downloadable bundle.
import hashlib
import json
import shutil
from pathlib import Path

if len(PUBLIC_RESULT_ZIPS) != 3:
    raise RuntimeError(f'Expected three result ZIPs, found {len(PUBLIC_RESULT_ZIPS)}: {PUBLIC_RESULT_ZIPS}')

bundle_root = Path('/content/qxern_v6_independent_audit_public_results')
if bundle_root.exists():
    shutil.rmtree(bundle_root)
bundle_root.mkdir(parents=True)

manifest = {
    'bundle': 'Qxern v6 public independent audit',
    'result_archives': [],
}
for item in PUBLIC_RESULT_ZIPS:
    src = Path(item)
    dst = bundle_root / src.name
    shutil.copy2(src, dst)
    digest = hashlib.sha256(dst.read_bytes()).hexdigest()
    manifest['result_archives'].append({
        'filename': dst.name,
        'sha256': digest,
        'size_bytes': dst.stat().st_size,
    })

(bundle_root / 'MANIFEST.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
final_zip = Path(shutil.make_archive(
    '/content/qxern_v6_independent_audit_public_results',
    'zip',
    root_dir=bundle_root,
))
print('Public result bundle:', final_zip)
print('SHA-256:', hashlib.sha256(final_zip.read_bytes()).hexdigest())

try:
    from google.colab import files
    files.download(str(final_zip))
except Exception:
    print('Download manually from:', final_zip)


Public result bundle created successfully.
SHA-256: 2fefa1e006f4644edf598aea26c72a0dfdf1420450f060c461686c220ddc88f9
